In [1]:
#data cleaning


import pandas as pd
import numpy as np
import os

# Main project directory
project_path = "/Users/simrannimje/Desktop/Collections_Analytics_Assignment"

# Raw and cleaned data directories
raw_path = os.path.join(project_path, "Data", "Raw")
cleaned_path = os.path.join(project_path, "Data", "Cleaned")

# Create Cleaned folder if it doesn't exist
os.makedirs(cleaned_path, exist_ok=True)

print("Data Cleaning Environment Ready!")
print("Raw Data Path:", raw_path)
print("Cleaned Data Path:", cleaned_path)

Data Cleaning Environment Ready!
Raw Data Path: /Users/simrannimje/Desktop/Collections_Analytics_Assignment/Data/Raw
Cleaned Data Path: /Users/simrannimje/Desktop/Collections_Analytics_Assignment/Data/Cleaned


In [2]:
# Load original payments dataset
payments_raw = pd.read_csv(
    os.path.join(raw_path, "payments.csv")
)

print("Original Payments Dataset")
print("Shape:", payments_raw.shape)

print("\nColumn Names:")
print(payments_raw.columns.tolist())

print("\nData Types:")
display(payments_raw.dtypes)

print("\nFirst 5 Records:")
display(payments_raw.head())

Original Payments Dataset
Shape: (25500, 9)

Column Names:
['payment_id', 'account_id', 'borrower_id', 'event_at', 'payment_reference', 'amount', 'payment_status', 'payment_method', 'provider_id']

Data Types:


payment_id               str
account_id               str
borrower_id              str
event_at                 str
payment_reference        str
amount               float64
payment_status           str
payment_method           str
provider_id              str
dtype: object


First 5 Records:


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
0,PAYMENT0000001,ACC0015539,BRW0011363,2026-02-27 01:28:12,TXN0000007457,22433.23,FAILED,CARD,VND0000001
1,PAYMENT0000002,ACC0004445,BRW0011306,2026-07-23 20:25:20,TXN0000030016,23295.11,FAILED,UPI,VND0000004
2,PAYMENT0000003,ACC0003853,BRW0002549,2026-01-11 21:27:46,TXN0000010466,118814.97,FAILED,NACH,VND0000013
3,PAYMENT0000004,ACC0017531,BRW0006618,2026-06-16 02:35:21,TXN0000031781,145082.17,REVERSED,CASH,VND0000013
4,PAYMENT0000005,ACC0001770,BRW0011492,2026-03-03 06:08:23,TXN0000059257,115148.46,SUCCESS,NETBANKING,VND0000013


In [3]:

#initial payment quality check

print("1. MISSING VALUES")
display(payments_raw.isnull().sum())

print("\n2. EXACT DUPLICATE ROWS")
print(payments_raw.duplicated().sum())

print("\n3. DUPLICATE PAYMENT IDs")
print(payments_raw["payment_id"].duplicated().sum())

print("\n4. UNIQUE PAYMENT STATUSES")
display(payments_raw["payment_status"].value_counts())

print("\n5. PAYMENT AMOUNT VALIDATION")

print("Zero amounts:",
      (payments_raw["amount"] == 0).sum())

print("Negative amounts:",
      (payments_raw["amount"] < 0).sum())

print("Missing amounts:",
      payments_raw["amount"].isnull().sum())

print("\n6. PAYMENT DATE VALIDATION")

print("Minimum date:", payments_raw["event_at"].min())
print("Maximum date:", payments_raw["event_at"].max())

1. MISSING VALUES


payment_id             0
account_id             0
borrower_id            0
event_at               0
payment_reference    382
amount                 0
payment_status         0
payment_method         0
provider_id            0
dtype: int64


2. EXACT DUPLICATE ROWS
486

3. DUPLICATE PAYMENT IDs
500

4. UNIQUE PAYMENT STATUSES


payment_status
SUCCESS     17880
FAILED       3744
PENDING      2592
REVERSED     1284
Name: count, dtype: int64


5. PAYMENT AMOUNT VALIDATION
Zero amounts: 0
Negative amounts: 0
Missing amounts: 0

6. PAYMENT DATE VALIDATION
Minimum date: 2026-01-01 00:14:40
Maximum date: 2026-08-08 23:50:23


In [4]:
# Check repeated payment IDs
duplicate_payments = payments_raw[
    payments_raw.duplicated(
        subset=["payment_id"],
        keep=False
    )
].sort_values("payment_id")

print("Records with repeated payment IDs:", len(duplicate_payments))

display(duplicate_payments.head(20))

# Check whether repeated IDs have conflicting attributes
payment_variations = (
    duplicate_payments.groupby("payment_id")
    .agg({
        "account_id": "nunique",
        "borrower_id": "nunique",
        "event_at": "nunique",
        "amount": "nunique",
        "payment_status": "nunique",
        "payment_method": "nunique"
    })
)

print("\nRepeated IDs with conflicting attributes:")
display(payment_variations[
    (payment_variations > 1).any(axis=1)
].head(20))

print(
    "\nTotal conflicting payment IDs:",
    ((payment_variations > 1).any(axis=1)).sum()
)

Records with repeated payment IDs: 1000


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
55,PAYMENT0000056,ACC0019495,BRW0009716,2026-02-24 09:33:29,TXN0000008612,77982.73,PENDING,NETBANKING,VND0000007
25035,PAYMENT0000056,ACC0019495,BRW0009716,2026-02-24 09:33:29,TXN0000008612,77982.73,PENDING,NETBANKING,VND0000007
75,PAYMENT0000076,ACC0027928,BRW0010869,2026-02-05 17:14:30,TXN0000006417,43712.91,PENDING,UPI,VND0000014
25408,PAYMENT0000076,ACC0027928,BRW0010869,2026-02-05 17:14:30,TXN0000006417,43712.91,PENDING,UPI,VND0000014
107,PAYMENT0000108,ACC0017047,BRW0009175,2026-06-09 08:17:45,TXN0000041404,102830.42,SUCCESS,NACH,VND0000003
25481,PAYMENT0000108,ACC0017047,BRW0009175,2026-06-09 08:17:45,TXN0000041404,102830.42,SUCCESS,NACH,VND0000003
148,PAYMENT0000149,ACC0018444,BRW0005741,2026-07-25 03:04:14,TXN0000066565,96766.12,SUCCESS,CASH,VND0000013
25099,PAYMENT0000149,ACC0018444,BRW0005741,2026-07-25 03:04:14,TXN0000066565,96766.12,SUCCESS,CASH,VND0000013
198,PAYMENT0000199,ACC0000056,BRW0001872,2026-05-12 22:49:22,TXN0000069967,81814.00,SUCCESS,NETBANKING,VND0000012
25330,PAYMENT0000199,ACC0000056,BRW0001872,2026-05-12 22:49:22,TXN0000069967,81814.00,SUCCESS,NETBANKING,VND0000012



Repeated IDs with conflicting attributes:


,account_id,borrower_id,event_at,amount,payment_status,payment_method
payment_id,,,,,,



Total conflicting payment IDs: 0


In [5]:
# Create a working copy
payments_clean = payments_raw.copy()

# Convert event_at to datetime
payments_clean["event_at"] = pd.to_datetime(
    payments_clean["event_at"],
    errors="coerce"
)

# Remove exact duplicate payment IDs
payments_clean = payments_clean.drop_duplicates(
    subset=["payment_id"],
    keep="first"
)

print("Original records:", len(payments_raw))
print("Cleaned records:", len(payments_clean))
print("Records removed:", len(payments_raw) - len(payments_clean))

print("\nRemaining duplicate payment IDs:")
print(payments_clean["payment_id"].duplicated().sum())

print("\nUpdated data types:")
display(payments_clean.dtypes)

Original records: 25500
Cleaned records: 25000
Records removed: 500

Remaining duplicate payment IDs:
0

Updated data types:


payment_id                      str
account_id                      str
borrower_id                     str
event_at             datetime64[us]
payment_reference               str
amount                      float64
payment_status                  str
payment_method                  str
provider_id                     str
dtype: object

In [6]:
# Validate cleaned payment dataset

print("Missing values:")
display(payments_clean.isnull().sum())

print("\nDuplicate payment IDs:")
print(payments_clean["payment_id"].duplicated().sum())

print("\nPayment status distribution:")
display(payments_clean["payment_status"].value_counts())

print("\nPayment amount summary:")
display(payments_clean["amount"].describe().round(2))

print("\nDate range:")
print("First:", payments_clean["event_at"].min())
print("Last:", payments_clean["event_at"].max())

Missing values:


payment_id             0
account_id             0
borrower_id            0
event_at               0
payment_reference    371
amount                 0
payment_status         0
payment_method         0
provider_id            0
dtype: int64


Duplicate payment IDs:
0

Payment status distribution:


payment_status
SUCCESS     17534
FAILED       3677
PENDING      2535
REVERSED     1254
Name: count, dtype: int64


Payment amount summary:


count     25000.00
mean      75155.69
std       43504.36
min         103.68
25%       37175.00
50%       75122.54
75%      112897.48
max      149993.55
Name: amount, dtype: float64


Date range:
First: 2026-01-01 00:14:40
Last: 2026-08-08 23:50:23


In [7]:
# Save cleaned payments dataset

output_file = os.path.join(
    cleaned_path,
    "payments_cleaned.csv"
)

payments_clean.to_csv(output_file, index=False)

print("Cleaned dataset saved successfully!")
print(output_file)

# Verify saved file
payments_check = pd.read_csv(output_file)

print("\nSaved dataset shape:", payments_check.shape)
print("Duplicate payment IDs:",
      payments_check["payment_id"].duplicated().sum())

Cleaned dataset saved successfully!
/Users/simrannimje/Desktop/Collections_Analytics_Assignment/Data/Cleaned/payments_cleaned.csv

Saved dataset shape: (25000, 9)
Duplicate payment IDs: 0


In [8]:
# Load raw accounts dataset
accounts_raw = pd.read_csv(
    os.path.join(raw_path, "accounts.csv")
)

print("Accounts dataset shape:", accounts_raw.shape)

print("\nData types:")
display(accounts_raw.dtypes)

print("\nMissing values:")
display(accounts_raw.isnull().sum())

print("\nFirst five records:")
display(accounts_raw.head())

Accounts dataset shape: (30000, 11)

Data types:


account_id                str
borrower_id               str
loan_type                 str
principal_amount      float64
outstanding_amount    float64
dpd                     int64
risk_segment              str
status                    str
opened_at                 str
timezone                  str
schema_version            str
dtype: object


Missing values:


account_id              0
borrower_id           455
loan_type               0
principal_amount        0
outstanding_amount      0
dpd                     0
risk_segment            0
status                  0
opened_at               0
timezone                0
schema_version          0
dtype: int64


First five records:


,account_id,borrower_id,loan_type,principal_amount,outstanding_amount,dpd,risk_segment,status,opened_at,timezone,schema_version
0,ACC0000001,BRW0010742,CONSUMER,603443.40,678074.03,15,MEDIUM,CLOSED,2025-11-11 04:37:00,Asia/Dubai,v1
1,ACC0000002,BRW0009382,BNPL,277190.14,464893.44,5,LOW,ACTIVE,2025-11-13 15:59:44,UTC,v3
2,ACC0000003,BRW0003966,CREDIT_CARD,628658.43,22565.37,60,NPA,WRITEOFF,2025-09-12 04:59:20,Asia/Dubai,v2
3,ACC0000004,BRW0001993,PERSONAL,76435.07,508427.73,30,LOW,CLOSED,2025-05-25 19:29:38,UTC,v2
4,ACC0000005,BRW0009976,CONSUMER,646797.08,563858.01,180,LOW,WRITEOFF,2025-09-07 14:59:13,Asia/Dubai,v2


In [9]:
# Create a working copy
accounts_clean = accounts_raw.copy()

# Convert opening date to datetime
accounts_clean["opened_at"] = pd.to_datetime(
    accounts_clean["opened_at"],
    errors="coerce"
)

# Check duplicate account IDs
print("Duplicate account IDs:",
      accounts_clean["account_id"].duplicated().sum())

# Check missing borrower IDs
print("Missing borrower IDs:",
      accounts_clean["borrower_id"].isnull().sum())

# Check invalid numeric values
print("\nNegative principal amounts:",
      (accounts_clean["principal_amount"] < 0).sum())

print("Negative outstanding amounts:",
      (accounts_clean["outstanding_amount"] < 0).sum())

print("Negative DPD values:",
      (accounts_clean["dpd"] < 0).sum())

# Check DPD categories
print("\nDPD values:")
display(accounts_clean["dpd"].value_counts().sort_index())

# Check opening date conversion
print("\nMissing opening dates after conversion:",
      accounts_clean["opened_at"].isnull().sum())

Duplicate account IDs: 0
Missing borrower IDs: 455

Negative principal amounts: 0
Negative outstanding amounts: 0
Negative DPD values: 0

DPD values:


dpd
0      2685
1      2713
5      2727
15     2736
30     2704
45     2744
60     2770
75     2741
90     2727
120    2759
180    2694
Name: count, dtype: int64


Missing opening dates after conversion: 0


In [10]:
# Flag accounts where outstanding exceeds principal
accounts_clean["outstanding_above_principal"] = (
    accounts_clean["outstanding_amount"] >
    accounts_clean["principal_amount"]
)

# Flag missing borrower IDs
accounts_clean["borrower_id_missing"] = (
    accounts_clean["borrower_id"].isnull()
)

print("Outstanding greater than principal:")
print(accounts_clean["outstanding_above_principal"].sum())

print("\nMissing borrower ID flag:")
print(accounts_clean["borrower_id_missing"].sum())

print("\nFlag distribution:")
display(
    accounts_clean[
        ["outstanding_above_principal", "borrower_id_missing"]
    ].sum()
)

Outstanding greater than principal:
13029

Missing borrower ID flag:
455

Flag distribution:


outstanding_above_principal    13029
borrower_id_missing              455
dtype: int64

In [11]:
# Save cleaned accounts dataset

output_file = os.path.join(
    cleaned_path,
    "accounts_cleaned.csv"
)

accounts_clean.to_csv(output_file, index=False)

print("Cleaned accounts dataset saved!")
print(output_file)

# Verify saved dataset
accounts_check = pd.read_csv(output_file)

print("\nSaved dataset shape:", accounts_check.shape)

print(
    "Duplicate account IDs:",
    accounts_check["account_id"].duplicated().sum()
)

print(
    "Missing borrower IDs:",
    accounts_check["borrower_id"].isnull().sum()
)

Cleaned accounts dataset saved!
/Users/simrannimje/Desktop/Collections_Analytics_Assignment/Data/Cleaned/accounts_cleaned.csv

Saved dataset shape: (30000, 13)
Duplicate account IDs: 0
Missing borrower IDs: 455


In [12]:
# Load raw borrowers dataset
borrowers_raw = pd.read_csv(
    os.path.join(raw_path, "borrowers.csv")
)

print("Borrowers dataset shape:", borrowers_raw.shape)

print("\nData types:")
display(borrowers_raw.dtypes)

print("\nMissing values:")
display(borrowers_raw.isnull().sum())

print("\nFirst five records:")
display(borrowers_raw.head())

Borrowers dataset shape: (30600, 8)

Data types:


borrower_id        str
name               str
phone          float64
email              str
city               str
created_at         str
updated_at         str
state              str
dtype: object


Missing values:


borrower_id      0
name             0
phone          614
email          895
city             0
created_at       0
updated_at       0
state            0
dtype: int64


First five records:


,borrower_id,name,phone,email,city,created_at,updated_at,state
0,BRW0001072,Rahul Verma,9.037419e+09,user9183@example.com,Bhubaneswar,2025-09-25 08:02:29,2026-07-25 22:25:48,Odisha
1,BRW0009288,Neha Singh,9.154913e+09,user9581@example.com,Gurugram,2025-06-29 04:32:01,2025-03-04 12:15:30,Haryana
2,BRW0007855,Vikram Shah,9.986755e+09,user19407@example.com,Hyderabad,2025-11-23 20:30:50,2025-01-07 21:07:17,Telangana
3,BRW0005267,Priya Mehta,9.145160e+09,user4619@example.com,Jaipur,2026-05-09 11:34:12,2025-04-24 21:35:48,Rajasthan
4,BRW0005197,Priya Mehta,9.655567e+09,user4597@example.com,Mumbai,2025-02-13 03:53:20,2026-03-29 19:42:02,Maharashtra


In [13]:
#before cleaning we will once check
# Create a working copy
borrowers_clean = borrowers_raw.copy()

# Convert phone numbers to nullable strings
borrowers_clean["phone"] = (
    borrowers_clean["phone"]
    .astype("Int64")
    .astype("string")
)

# Convert timestamps
borrowers_clean["created_at"] = pd.to_datetime(
    borrowers_clean["created_at"], errors="coerce"
)

borrowers_clean["updated_at"] = pd.to_datetime(
    borrowers_clean["updated_at"], errors="coerce"
)

# Check exact duplicate records
print("Exact duplicate rows:",
      borrowers_clean.duplicated().sum())

# Check repeated borrower IDs
print("Repeated borrower ID rows:",
      borrowers_clean["borrower_id"].duplicated().sum())

# Check timestamp inconsistencies
invalid_dates = borrowers_clean[
    borrowers_clean["updated_at"] < borrowers_clean["created_at"]
]

print("Updated before created:", len(invalid_dates))

# Check missing contact information
print("\nMissing phone:", borrowers_clean["phone"].isna().sum())
print("Missing email:", borrowers_clean["email"].isna().sum())

# Display examples of timestamp inconsistencies
display(
    invalid_dates[
        ["borrower_id", "created_at", "updated_at"]
    ].head(10)
)

Exact duplicate rows: 600
Repeated borrower ID rows: 19585
Updated before created: 15354

Missing phone: 614
Missing email: 895


,borrower_id,created_at,updated_at
1,BRW0009288,2025-06-29 04:32:01,2025-03-04 12:15:30
2,BRW0007855,2025-11-23 20:30:50,2025-01-07 21:07:17
3,BRW0005267,2026-05-09 11:34:12,2025-04-24 21:35:48
7,BRW0008369,2026-05-27 08:50:59,2025-09-20 02:27:08
8,BRW0002418,2026-05-27 06:05:11,2025-07-19 15:06:52
9,BRW0001131,2025-11-23 01:27:02,2025-03-21 22:18:48
11,BRW0011708,2026-04-08 06:45:08,2025-02-07 01:14:56
13,BRW0009134,2026-02-10 17:43:22,2025-01-19 21:07:39
14,BRW0008610,2026-03-29 04:42:45,2025-07-11 05:04:23
16,BRW0006159,2025-11-25 04:20:45,2025-11-24 18:11:12


In [14]:
# Remove exact duplicate records only
borrowers_clean = borrowers_clean.drop_duplicates()

# Create a flag for missing phone numbers
borrowers_clean["phone_missing"] = (
    borrowers_clean["phone"].isna()
)

# Create a flag for missing email addresses
borrowers_clean["email_missing"] = (
    borrowers_clean["email"].isna()
)

# Flag inconsistent timestamps
borrowers_clean["timestamp_inconsistent"] = (
    borrowers_clean["updated_at"] <
    borrowers_clean["created_at"]
)

# Validate the cleaned dataset
print("Original rows:", len(borrowers_raw))
print("Cleaned rows:", len(borrowers_clean))
print("Rows removed:", len(borrowers_raw) - len(borrowers_clean))

print("\nRemaining exact duplicates:",
      borrowers_clean.duplicated().sum())

print("Repeated borrower IDs:",
      borrowers_clean["borrower_id"].duplicated().sum())

print("Missing phone flags:",
      borrowers_clean["phone_missing"].sum())

print("Missing email flags:",
      borrowers_clean["email_missing"].sum())

print("Timestamp inconsistency flags:",
      borrowers_clean["timestamp_inconsistent"].sum())

Original rows: 30600
Cleaned rows: 30000
Rows removed: 600

Remaining exact duplicates: 0
Repeated borrower IDs: 18985
Missing phone flags: 604
Missing email flags: 880
Timestamp inconsistency flags: 15063


In [15]:
# Save cleaned borrowers dataset
output_file = os.path.join(
    cleaned_path, "borrowers_cleaned.csv"
)

borrowers_clean.to_csv(output_file, index=False)

print("Cleaned borrowers dataset saved!")
print(output_file)

# Verify saved file
borrowers_check = pd.read_csv(output_file)

print("\nSaved dataset shape:", borrowers_check.shape)
print("Exact duplicate rows:", borrowers_check.duplicated().sum())
print("Repeated borrower IDs:",
      borrowers_check["borrower_id"].duplicated().sum())

Cleaned borrowers dataset saved!
/Users/simrannimje/Desktop/Collections_Analytics_Assignment/Data/Cleaned/borrowers_cleaned.csv

Saved dataset shape: (30000, 11)
Exact duplicate rows: 0
Repeated borrower IDs: 18985


In [16]:
# Load raw agents dataset
agents_raw = pd.read_csv(
    os.path.join(raw_path, "agents.csv")
)

print("Agents dataset shape:", agents_raw.shape)

print("\nData types:")
display(agents_raw.dtypes)

print("\nMissing values:")
display(agents_raw.isnull().sum())

print("\nFirst five records:")
display(agents_raw.head())

Agents dataset shape: (30000, 8)

Data types:


agent_id         str
employee_code    str
agent_name       str
vendor_id        str
team             str
status           str
joined_at        str
updated_at       str
dtype: object


Missing values:


agent_id         0
employee_code    0
agent_name       0
vendor_id        0
team             0
status           0
joined_at        0
updated_at       0
dtype: int64


First five records:


,agent_id,employee_code,agent_name,vendor_id,team,status,joined_at,updated_at
0,AGT0000760,EMP00323,Vikram Shah,VND0000012,T3,INACTIVE,2025-06-19 20:30:53,2025-12-11 03:40:22
1,AGT0000171,EMP00079,Pooja Nair,VND0000012,T1,INACTIVE,2025-08-26 11:06:26,2025-10-11 08:30:33
2,AGT0000766,EMP00030,Sneha Das,VND0000002,FIELD,SUSPENDED,2024-05-30 11:23:06,2026-05-16 03:02:02
3,AGT0000031,EMP00334,Aarav Sharma,VND0000007,T1,INACTIVE,2024-10-11 18:58:28,2025-10-16 13:31:21
4,AGT0000669,EMP00014,Aarav Sharma,VND0000001,T3,SUSPENDED,2024-09-01 19:55:06,2026-07-04 03:33:11


In [17]:
# Create a working copy
agents_clean = agents_raw.copy()

# Convert timestamps
agents_clean["joined_at"] = pd.to_datetime(
    agents_clean["joined_at"], errors="coerce"
)

agents_clean["updated_at"] = pd.to_datetime(
    agents_clean["updated_at"], errors="coerce"
)

# Check exact duplicates
print("Exact duplicate rows:",
      agents_clean.duplicated().sum())

# Check unique and repeated agent IDs
print("\nTotal records:", len(agents_clean))
print("Unique agent IDs:", agents_clean["agent_id"].nunique())
print("Repeated agent ID rows:",
      agents_clean["agent_id"].duplicated().sum())

# Check unique employee codes
print("Unique employee codes:",
      agents_clean["employee_code"].nunique())

# Check timestamp inconsistencies
invalid_dates = agents_clean[
    agents_clean["updated_at"] < agents_clean["joined_at"]
]

print("\nUpdated before joined:", len(invalid_dates))
print("Affected agent IDs:",
      invalid_dates["agent_id"].nunique())

# Display sample repeated agent IDs
display(
    agents_clean[
        agents_clean["agent_id"].duplicated(keep=False)
    ].sort_values("agent_id").head(15)
)

Exact duplicate rows: 0

Total records: 30000
Unique agent IDs: 1000
Repeated agent ID rows: 29000
Unique employee codes: 1099

Updated before joined: 4078
Affected agent IDs: 975


,agent_id,employee_code,agent_name,vendor_id,team,status,joined_at,updated_at
278,AGT0000001,EMP01065,Neha Singh,VND0000007,T3,ACTIVE,2024-05-12 03:20:38,2026-04-14 22:45:46
3016,AGT0000001,EMP00285,Sneha Das,VND0000013,DIGITAL,INACTIVE,2025-05-02 11:13:31,2025-02-14 12:38:08
4511,AGT0000001,EMP00113,Amit Kumar,VND0000015,T1,SUSPENDED,2025-08-03 12:02:41,2026-05-03 03:37:58
5522,AGT0000001,EMP00196,Aarav Sharma,VND0000008,T1,ACTIVE,2024-08-21 06:20:50,2026-03-16 07:28:16
6520,AGT0000001,EMP00632,Ananya Rao,VND0000007,DIGITAL,ACTIVE,2024-08-02 00:51:56,2025-08-30 06:00:31
7200,AGT0000001,EMP00117,Pooja Nair,VND0000009,T3,ACTIVE,2024-02-03 19:49:59,2026-04-17 04:53:24
8769,AGT0000001,EMP00861,Aarav Sharma,VND0000015,T3,INACTIVE,2025-03-16 08:57:58,2026-05-11 23:40:59
8967,AGT0000001,EMP01082,Neha Singh,VND0000013,T1,INACTIVE,2025-05-02 01:36:54,2025-04-25 02:56:44
9420,AGT0000001,EMP00723,Rahul Verma,VND0000013,T2,ACTIVE,2025-03-04 14:23:32,2025-10-11 06:30:44
9554,AGT0000001,EMP00883,Sneha Das,VND0000010,T3,INACTIVE,2025-09-15 12:18:43,2025-02-08 10:18:20


In [19]:
# Flag timestamp inconsistencies
agents_clean["timestamp_inconsistent"] = (
    agents_clean["updated_at"] < agents_clean["joined_at"]
)

# Check identity mapping variations
agent_variations = (
    agents_clean.groupby("agent_id")
    .agg(
        employee_codes=("employee_code", "nunique"),
        agent_names=("agent_name", "nunique"),
        vendors=("vendor_id", "nunique"),
        teams=("team", "nunique"),
        statuses=("status", "nunique")
    )
    .reset_index()
)

# Summarise identity inconsistencies
print("Agent IDs with multiple employee codes:",
      (agent_variations["employee_codes"] > 1).sum())

print("Agent IDs with multiple names:",
      (agent_variations["agent_names"] > 1).sum())

print("Agent IDs with multiple vendors:",
      (agent_variations["vendors"] > 1).sum())

print("Agent IDs with multiple teams:",
      (agent_variations["teams"] > 1).sum())

print("Agent IDs with multiple statuses:",
      (agent_variations["statuses"] > 1).sum())

print("\nTimestamp inconsistency flags:",
      agents_clean["timestamp_inconsistent"].sum())

Agent IDs with multiple employee codes: 1000
Agent IDs with multiple names: 1000
Agent IDs with multiple vendors: 1000
Agent IDs with multiple teams: 1000
Agent IDs with multiple statuses: 1000

Timestamp inconsistency flags: 4078


In [20]:
# Save cleaned agents dataset
output_file = os.path.join(
    cleaned_path, "agents_cleaned.csv"
)

agents_clean.to_csv(output_file, index=False)

print("Cleaned agents dataset saved!")
print(output_file)

# Verify saved file
agents_check = pd.read_csv(output_file)

print("\nSaved dataset shape:", agents_check.shape)
print("Exact duplicate rows:",
      agents_check.duplicated().sum())
print("Unique agent IDs:",
      agents_check["agent_id"].nunique())
print("Timestamp inconsistency flags:",
      agents_check["timestamp_inconsistent"].sum())

Cleaned agents dataset saved!
/Users/simrannimje/Desktop/Collections_Analytics_Assignment/Data/Cleaned/agents_cleaned.csv

Saved dataset shape: (30000, 9)
Exact duplicate rows: 0
Unique agent IDs: 1000
Timestamp inconsistency flags: 4078


In [21]:
# Load agent sessions dataset
sessions_raw = pd.read_csv(
    os.path.join(raw_path, "agent_sessions.csv")
)

print("Dataset shape:", sessions_raw.shape)

print("\nData types:")
display(sessions_raw.dtypes)

print("\nMissing values:")
display(sessions_raw.isnull().sum())

print("\nFirst five records:")
display(sessions_raw.head())

Dataset shape: (15000, 7)

Data types:


session_id    str
agent_id      str
login_at      str
channel       str
device_id     str
timezone      str
logout_at     str
dtype: object


Missing values:


session_id    0
agent_id      0
login_at      0
channel       0
device_id     0
timezone      0
logout_at     0
dtype: int64


First five records:


,session_id,agent_id,login_at,channel,device_id,timezone,logout_at
0,SES0000001,AGT0000117,2026-08-06 13:33:27,VOICE,DEV00243,Asia/Kolkata,2026-08-06 18:41:13
1,SES0000002,AGT0000847,2026-05-27 02:02:28,FIELD,DEV01359,UTC,2026-05-27 09:02:31
2,SES0000003,AGT0000989,2026-08-01 20:33:44,FIELD,DEV01432,Asia/Kolkata,2026-08-02 05:53:43
3,SES0000004,AGT0000961,2026-01-13 19:50:52,FIELD,DEV00402,Asia/Kolkata,2026-01-13 22:20:49
4,SES0000005,AGT0000280,2026-03-03 15:05:04,VOICE,DEV01288,UTC,2026-03-03 18:19:09


In [22]:
# Create a working copy
sessions_clean = sessions_raw.copy()

# Convert timestamps
sessions_clean["login_at"] = pd.to_datetime(
    sessions_clean["login_at"], errors="coerce"
)

sessions_clean["logout_at"] = pd.to_datetime(
    sessions_clean["logout_at"], errors="coerce"
)

# Check duplicate records
print("Exact duplicate rows:",
      sessions_clean.duplicated().sum())

print("Duplicate session IDs:",
      sessions_clean["session_id"].duplicated().sum())

# Check unique agents and devices
print("\nUnique agent IDs:",
      sessions_clean["agent_id"].nunique())

print("Unique devices:",
      sessions_clean["device_id"].nunique())

# Calculate session duration in hours
sessions_clean["session_duration_hours"] = (
    sessions_clean["logout_at"] -
    sessions_clean["login_at"]
).dt.total_seconds() / 3600

# Check session duration
print("\nSession duration summary:")
display(sessions_clean["session_duration_hours"].describe())

# Identify invalid durations
invalid_sessions = sessions_clean[
    sessions_clean["session_duration_hours"] <= 0
]

print("Zero or negative duration sessions:",
      len(invalid_sessions))

# Check timezone distribution
print("\nTimezone distribution:")
display(sessions_clean["timezone"].value_counts())

# Check channel distribution
print("\nChannel distribution:")
display(sessions_clean["channel"].value_counts())


Exact duplicate rows: 0
Duplicate session IDs: 0

Unique agent IDs: 1000
Unique devices: 1500

Session duration summary:


count    15000.000000
mean         5.258059
std          2.739664
min          0.500000
25%          2.876250
50%          5.236389
75%          7.633889
max          9.998333
Name: session_duration_hours, dtype: float64

Zero or negative duration sessions: 0

Timezone distribution:


timezone
Asia/Kolkata    7506
UTC             7494
Name: count, dtype: int64


Channel distribution:


channel
WHATSAPP    3810
FIELD       3796
VOICE       3740
SMS         3654
Name: count, dtype: int64

In [23]:
# Add a flag for session duration validation
sessions_clean["valid_session_duration"] = (
    sessions_clean["session_duration_hours"] > 0
)

# Save cleaned dataset
output_file = os.path.join(
    cleaned_path, "agent_sessions_cleaned.csv"
)

sessions_clean.to_csv(output_file, index=False)

print("Cleaned agent sessions dataset saved!")
print(output_file)

# Verify saved file
sessions_check = pd.read_csv(output_file)

print("\nSaved dataset shape:", sessions_check.shape)
print("Duplicate session IDs:",
      sessions_check["session_id"].duplicated().sum())
print("Invalid session durations:",
      (sessions_check["valid_session_duration"] == False).sum())

Cleaned agent sessions dataset saved!
/Users/simrannimje/Desktop/Collections_Analytics_Assignment/Data/Cleaned/agent_sessions_cleaned.csv

Saved dataset shape: (15000, 9)
Duplicate session IDs: 0
Invalid session durations: 0


In [24]:
# Load campaigns dataset
campaigns_raw = pd.read_csv(
    os.path.join(raw_path, "campaigns.csv")
)

print("Dataset shape:", campaigns_raw.shape)

print("\nData types:")
display(campaigns_raw.dtypes)

print("\nMissing values:")
display(campaigns_raw.isnull().sum())

print("\nFirst five records:")
display(campaigns_raw.head())

Dataset shape: (120, 7)

Data types:


campaign_id          str
campaign_name        str
channel              str
strategy_version     str
start_at             str
target_definition    str
end_at               str
dtype: object


Missing values:


campaign_id          0
campaign_name        0
channel              0
strategy_version     0
start_at             0
target_definition    0
end_at               0
dtype: int64


First five records:


,campaign_id,campaign_name,channel,strategy_version,start_at,target_definition,end_at
0,CMP0000001,DIGITAL_FOLLOWUP,FIELD,legacy,2026-02-17 06:56:01,DPD>=30,2026-04-24 06:56:01
1,CMP0000002,BOUNCE,MIXED,v2,2026-04-30 17:51:31,DPD>=60,2026-05-18 17:51:31
2,CMP0000003,30DPD_W1,MIXED,v1,2026-04-11 16:02:51,DPD>=30,2026-05-18 16:02:51
3,CMP0000004,BOUNCE,VOICE,legacy,2026-04-28 06:15:30,DPD>=60,2026-05-29 06:15:30
4,CMP0000005,60DPD_INTENT,WHATSAPP,legacy,2026-05-04 04:16:21,DPD>=60,2026-07-08 04:16:21


In [25]:
# Create a working copy
campaigns_clean = campaigns_raw.copy()

# Convert campaign dates
campaigns_clean["start_at"] = pd.to_datetime(
    campaigns_clean["start_at"], errors="coerce"
)

campaigns_clean["end_at"] = pd.to_datetime(
    campaigns_clean["end_at"], errors="coerce"
)

# Check duplicate campaign IDs
print("Duplicate campaign IDs:",
      campaigns_clean["campaign_id"].duplicated().sum())

# Check repeated campaign names
print("Unique campaign names:",
      campaigns_clean["campaign_name"].nunique())

print("Unique strategy versions:",
      campaigns_clean["strategy_version"].nunique())

print("Unique channels:",
      campaigns_clean["channel"].nunique())

# Check campaign duration
campaigns_clean["campaign_duration_days"] = (
    campaigns_clean["end_at"] -
    campaigns_clean["start_at"]
).dt.total_seconds() / 86400

print("\nCampaign duration summary:")
display(campaigns_clean["campaign_duration_days"].describe())

# Identify invalid campaign dates
invalid_campaigns = campaigns_clean[
    campaigns_clean["end_at"] <= campaigns_clean["start_at"]
]

print("Invalid campaign date ranges:",
      len(invalid_campaigns))

# Check campaign combinations
print("\nCampaign name, channel and strategy combinations:")
display(
    campaigns_clean.groupby(
        ["campaign_name", "channel", "strategy_version"]
    ).size().reset_index(name="records").head(15)
)

# Check target definitions
print("\nTarget definitions:")
display(campaigns_clean["target_definition"].value_counts())

Duplicate campaign IDs: 0
Unique campaign names: 5
Unique strategy versions: 4
Unique channels: 5

Campaign duration summary:


count    120.000000
mean      42.458333
std       24.725687
min        6.000000
25%       20.000000
50%       40.000000
75%       65.250000
max       89.000000
Name: campaign_duration_days, dtype: float64

Invalid campaign date ranges: 0

Campaign name, channel and strategy combinations:


,campaign_name,channel,strategy_version,records
0,30DPD_W1,FIELD,legacy,3
1,30DPD_W1,FIELD,v1,2
2,30DPD_W1,FIELD,v2,1
3,30DPD_W1,FIELD,v3,1
4,30DPD_W1,MIXED,legacy,2
5,30DPD_W1,MIXED,v1,2
6,30DPD_W1,MIXED,v2,1
7,30DPD_W1,MIXED,v3,2
8,30DPD_W1,SMS,legacy,2
9,30DPD_W1,SMS,v1,1



Target definitions:


target_definition
DPD>=60           31
HIGH_RISK         25
PROMISE_BROKEN    24
DPD>=30           22
NPA               18
Name: count, dtype: int64

In [26]:
# Save cleaned campaigns dataset
output_file = os.path.join(
    cleaned_path, "campaigns_cleaned.csv"
)

campaigns_clean.to_csv(output_file, index=False)

print("Cleaned campaigns dataset saved!")
print(output_file)

# Verify saved file
campaigns_check = pd.read_csv(output_file)

print("\nSaved dataset shape:", campaigns_check.shape)
print("Duplicate campaign IDs:",
      campaigns_check["campaign_id"].duplicated().sum())

Cleaned campaigns dataset saved!
/Users/simrannimje/Desktop/Collections_Analytics_Assignment/Data/Cleaned/campaigns_cleaned.csv

Saved dataset shape: (120, 8)
Duplicate campaign IDs: 0


In [27]:
# Load daily targeting dataset
targeting_raw = pd.read_csv(
    os.path.join(raw_path, "daily_targeting.csv")
)

print("Dataset shape:", targeting_raw.shape)

print("\nData types:")
display(targeting_raw.dtypes)

print("\nMissing values:")
display(targeting_raw.isnull().sum())

print("\nFirst five records:")
display(targeting_raw.head())

Dataset shape: (45000, 7)

Data types:


target_id                str
account_id               str
campaign_id              str
target_date              str
priority               int64
recommended_channel      str
status                   str
dtype: object


Missing values:


target_id              0
account_id             0
campaign_id            0
target_date            0
priority               0
recommended_channel    0
status                 0
dtype: int64


First five records:


,target_id,account_id,campaign_id,target_date,priority,recommended_channel,status
0,TGT0000001,ACC0028555,CMP0000103,2026-04-22,4,FIELD,QUEUED
1,TGT0000002,ACC0007194,CMP0000100,2026-08-06,10,SMS,EXPIRED
2,TGT0000003,ACC0029550,CMP0000074,2026-05-20,5,SMS,CONTACTED
3,TGT0000004,ACC0012329,CMP0000046,2026-07-30,10,WHATSAPP,QUEUED
4,TGT0000005,ACC0018387,CMP0000118,2026-07-10,7,WHATSAPP,EXPIRED


In [28]:
# Create a working copy
targeting_clean = targeting_raw.copy()

# Convert target date
targeting_clean["target_date"] = pd.to_datetime(
    targeting_clean["target_date"], errors="coerce"
)

# Check duplicate records
print("Exact duplicate rows:",
      targeting_clean.duplicated().sum())

print("Duplicate target IDs:",
      targeting_clean["target_id"].duplicated().sum())

# Check unique accounts and campaigns
print("\nUnique accounts:",
      targeting_clean["account_id"].nunique())

print("Unique campaigns:",
      targeting_clean["campaign_id"].nunique())

# Check priority range
print("\nPriority summary:")
display(targeting_clean["priority"].describe())

print("Priority values:")
display(targeting_clean["priority"].value_counts().sort_index())

# Check targeting dates
print("\nTargeting date range:")
print(targeting_clean["target_date"].min())
print(targeting_clean["target_date"].max())

# Check statuses
print("\nTargeting statuses:")
display(targeting_clean["status"].value_counts())

# Check recommended channels
print("\nRecommended channels:")
display(targeting_clean["recommended_channel"].value_counts())

# Validate account references
print("\nAccounts not found:",
      (~targeting_clean["account_id"].isin(
          accounts_clean["account_id"]
      )).sum())

# Validate campaign references
print("Campaigns not found:",
      (~targeting_clean["campaign_id"].isin(
          campaigns_clean["campaign_id"]
      )).sum())

Exact duplicate rows: 0
Duplicate target IDs: 0

Unique accounts: 23344
Unique campaigns: 120

Priority summary:


count    45000.000000
mean         5.505600
std          2.877085
min          1.000000
25%          3.000000
50%          6.000000
75%          8.000000
max         10.000000
Name: priority, dtype: float64

Priority values:


priority
1     4531
2     4539
3     4520
4     4358
5     4401
6     4432
7     4641
8     4587
9     4539
10    4452
Name: count, dtype: int64


Targeting date range:
2026-01-01 00:00:00
2026-08-08 00:00:00

Targeting statuses:


status
EXPIRED      11371
CONTACTED    11254
QUEUED       11202
SKIPPED      11173
Name: count, dtype: int64


Recommended channels:


recommended_channel
FIELD       11365
SMS         11221
WHATSAPP    11212
VOICE       11202
Name: count, dtype: int64


Accounts not found: 0
Campaigns not found: 0


In [29]:
# Save cleaned targeting dataset
output_file = os.path.join(
    cleaned_path, "daily_targeting_cleaned.csv"
)

targeting_clean.to_csv(output_file, index=False)

print("Cleaned targeting dataset saved!")
print(output_file)

# Verify saved file
targeting_check = pd.read_csv(output_file)

print("\nSaved dataset shape:", targeting_check.shape)
print("Duplicate target IDs:",
      targeting_check["target_id"].duplicated().sum())
print("Missing values:", targeting_check.isnull().sum().sum())


Cleaned targeting dataset saved!
/Users/simrannimje/Desktop/Collections_Analytics_Assignment/Data/Cleaned/daily_targeting_cleaned.csv

Saved dataset shape: (45000, 7)
Duplicate target IDs: 0
Missing values: 0


In [30]:
# Load calls dataset
calls_raw = pd.read_csv(
    os.path.join(raw_path, "calls.csv")
)

print("Dataset shape:", calls_raw.shape)

print("\nData types:")
display(calls_raw.dtypes)

print("\nMissing values:")
display(calls_raw.isnull().sum())

print("\nFirst five records:")
display(calls_raw.head())

Dataset shape: (91350, 11)

Data types:


call_id           str
account_id        str
borrower_id       str
event_at          str
agent_id          str
campaign_id       str
direction         str
vendor_id         str
call_status       str
duration_sec    int64
timezone          str
dtype: object


Missing values:


call_id            0
account_id         0
borrower_id        0
event_at           0
agent_id        1827
campaign_id        0
direction          0
vendor_id          0
call_status        0
duration_sec       0
timezone           0
dtype: int64


First five records:


,call_id,account_id,borrower_id,event_at,agent_id,campaign_id,direction,vendor_id,call_status,duration_sec,timezone
0,CALL0000001,ACC0011505,BRW0007139,2026-07-15 15:36:22,AGT0000955,CMP0000015,OUTBOUND,VND0000013,NO_ANSWER,601,Asia/Dubai
1,CALL0000002,ACC0002025,BRW0006253,2026-06-10 06:48:27,AGT0000853,CMP0000060,OUTBOUND,VND0000001,ANSWERED,224,Asia/Kolkata
2,CALL0000003,ACC0013375,BRW0007663,2026-04-07 00:35:35,AGT0000525,CMP0000060,OUTBOUND,VND0000008,FAILED,7,Asia/Dubai
3,CALL0000004,ACC0021534,BRW0008873,2026-02-12 14:16:57,AGT0000945,CMP0000060,OUTBOUND,VND0000001,VOICEMAIL,896,Asia/Dubai
4,CALL0000005,ACC0018502,BRW0004996,2026-05-24 15:33:12,AGT0000966,CMP0000053,OUTBOUND,VND0000006,ANSWERED,711,Asia/Kolkata


In [31]:
# Create a working copy
calls_clean = calls_raw.copy()

# Convert event timestamp
calls_clean["event_at"] = pd.to_datetime(
    calls_clean["event_at"], errors="coerce"
)

# Check duplicate records
print("Exact duplicate rows:",
      calls_clean.duplicated().sum())

print("Duplicate call IDs:",
      calls_clean["call_id"].duplicated().sum())

# Check missing agent IDs
print("\nMissing agent IDs:",
      calls_clean["agent_id"].isna().sum())

# Check call duration
print("\nCall duration summary:")
display(calls_clean["duration_sec"].describe())

print("Zero-duration calls:",
      (calls_clean["duration_sec"] == 0).sum())

print("Negative-duration calls:",
      (calls_clean["duration_sec"] < 0).sum())

# Check call statuses
print("\nCall statuses:")
display(calls_clean["call_status"].value_counts())

# Check call directions
print("\nCall directions:")
display(calls_clean["direction"].value_counts())

# Check timezones
print("\nTimezones:")
display(calls_clean["timezone"].value_counts())

# Validate account and campaign references
print("\nAccounts not found:",
      (~calls_clean["account_id"].isin(
          accounts_clean["account_id"]
      )).sum())

print("Campaigns not found:",
      (~calls_clean["campaign_id"].isin(
          campaigns_clean["campaign_id"]
      )).sum())

Exact duplicate rows: 1271
Duplicate call IDs: 1350

Missing agent IDs: 1827

Call duration summary:


count    91350.000000
mean       451.158905
std        259.019217
min          0.000000
25%        228.000000
50%        452.000000
75%        676.000000
max        899.000000
Name: duration_sec, dtype: float64

Zero-duration calls: 99
Negative-duration calls: 0

Call statuses:


call_status
NO_ANSWER    18363
BUSY         18330
FAILED       18276
VOICEMAIL    18235
ANSWERED     18146
Name: count, dtype: int64


Call directions:


direction
OUTBOUND    84151
INBOUND      7199
Name: count, dtype: int64


Timezones:


timezone
Asia/Kolkata    30485
Asia/Dubai      30464
UTC             30401
Name: count, dtype: int64


Accounts not found: 0
Campaigns not found: 0


In [32]:
# Remove only verified exact duplicate rows
calls_clean = calls_clean.drop_duplicates()

# Flag missing agent IDs
calls_clean["agent_id_missing"] = (
    calls_clean["agent_id"].isna()
)

# Flag zero-duration calls
calls_clean["zero_duration"] = (
    calls_clean["duration_sec"] == 0
)

# Flag repeated call IDs for further investigation
calls_clean["repeated_call_id"] = (
    calls_clean["call_id"].duplicated(keep=False)
)

# Validate cleaned dataset
print("Original rows:", len(calls_raw))
print("Cleaned rows:", len(calls_clean))
print("Rows removed:", len(calls_raw) - len(calls_clean))

print("\nRemaining exact duplicates:",
      calls_clean.duplicated().sum())

print("Repeated call IDs:",
      calls_clean["call_id"].duplicated().sum())

print("Missing agent ID flags:",
      calls_clean["agent_id_missing"].sum())

print("Zero-duration flags:",
      calls_clean["zero_duration"].sum())

print("Repeated call ID flags:",
      calls_clean["repeated_call_id"].sum())

Original rows: 91350
Cleaned rows: 90079
Rows removed: 1271

Remaining exact duplicates: 0
Repeated call IDs: 79
Missing agent ID flags: 1827
Zero-duration flags: 99
Repeated call ID flags: 158


In [33]:
# Save cleaned calls dataset
output_file = os.path.join(
    cleaned_path, "calls_cleaned.csv"
)

calls_clean.to_csv(output_file, index=False)

print("Cleaned calls dataset saved!")
print(output_file)

# Verify saved file
calls_check = pd.read_csv(output_file)

print("\nSaved dataset shape:", calls_check.shape)
print("Exact duplicate rows:", calls_check.duplicated().sum())
print("Repeated call IDs:",
      calls_check["call_id"].duplicated().sum())
print("Missing agent ID flags:",
      calls_check["agent_id_missing"].sum())

Cleaned calls dataset saved!
/Users/simrannimje/Desktop/Collections_Analytics_Assignment/Data/Cleaned/calls_cleaned.csv

Saved dataset shape: (90079, 14)
Exact duplicate rows: 0
Repeated call IDs: 79
Missing agent ID flags: 1827


In [34]:
# Load call attempts dataset
attempts_raw = pd.read_csv(
    os.path.join(raw_path, "call_attempts.csv")
)

print("Dataset shape:", attempts_raw.shape)

print("\nData types:")
display(attempts_raw.dtypes)

print("\nMissing values:")
display(attempts_raw.isnull().sum())

print("\nFirst five records:")
display(attempts_raw.head())

Dataset shape: (120000, 9)

Data types:


attempt_id          str
account_id          str
borrower_id         str
event_at            str
call_id             str
agent_id            str
attempt_no        int64
vendor_id           str
attempt_status      str
dtype: object


Missing values:


attempt_id           0
account_id           0
borrower_id          0
event_at             0
call_id              0
agent_id             0
attempt_no           0
vendor_id         2400
attempt_status       0
dtype: int64


First five records:


,attempt_id,account_id,borrower_id,event_at,call_id,agent_id,attempt_no,vendor_id,attempt_status
0,ATTEMPT0000001,ACC0009527,BRW0000705,2026-03-10 21:31:13,CALL0019052,AGT0000685,5,VND0000013,FAILED
1,ATTEMPT0000002,ACC0007960,BRW0002732,2026-05-01 20:55:02,CALL0011950,AGT0000472,4,VND0000006,CONNECTED
2,ATTEMPT0000003,ACC0016476,BRW0002174,2026-06-05 20:57:37,CALL0017411,AGT0000035,1,VND0000015,RINGING
3,ATTEMPT0000004,ACC0011111,BRW0003738,2026-05-02 06:08:39,CALL0068999,AGT0000277,3,VND0000010,FAILED
4,ATTEMPT0000005,ACC0010343,BRW0003352,2026-05-27 03:48:23,CALL0049585,AGT0000346,5,VND0000008,CONNECTED


In [35]:
# Create a working copy
attempts_clean = attempts_raw.copy()

# Convert event timestamp
attempts_clean["event_at"] = pd.to_datetime(
    attempts_clean["event_at"], errors="coerce"
)

# Check duplicates
print("Exact duplicate rows:",
      attempts_clean.duplicated().sum())

print("Duplicate attempt IDs:",
      attempts_clean["attempt_id"].duplicated().sum())

# Check unique entities
print("\nUnique accounts:",
      attempts_clean["account_id"].nunique())

print("Unique call IDs:",
      attempts_clean["call_id"].nunique())

print("Unique agents:",
      attempts_clean["agent_id"].nunique())

# Check attempt number
print("\nAttempt number summary:")
display(attempts_clean["attempt_no"].describe())

print("Attempt number distribution:")
display(attempts_clean["attempt_no"].value_counts().sort_index())

# Check attempt statuses
print("\nAttempt statuses:")
display(attempts_clean["attempt_status"].value_counts())

# Check vendor IDs
print("\nMissing vendor IDs:",
      attempts_clean["vendor_id"].isna().sum())

# Validate account references
print("\nAccounts not found:",
      (~attempts_clean["account_id"].isin(
          accounts_clean["account_id"]
      )).sum())

# Validate call references
print("Call IDs not found in cleaned calls:",
      (~attempts_clean["call_id"].isin(
          calls_clean["call_id"]
      )).sum())

# Validate agent references
print("Agent IDs not found:",
      (~attempts_clean["agent_id"].isin(
          agents_clean["agent_id"]
      )).sum())

Exact duplicate rows: 0
Duplicate attempt IDs: 0

Unique accounts: 29451
Unique call IDs: 66244
Unique agents: 1000

Attempt number summary:


count    120000.000000
mean          4.002458
std           1.999951
min           1.000000
25%           2.000000
50%           4.000000
75%           6.000000
max           7.000000
Name: attempt_no, dtype: float64

Attempt number distribution:


attempt_no
1    17089
2    17100
3    17201
4    17252
5    16952
6    17207
7    17199
Name: count, dtype: int64


Attempt statuses:


attempt_status
RINGING      24087
NO_ANSWER    24042
BUSY         24037
CONNECTED    24023
FAILED       23811
Name: count, dtype: int64


Missing vendor IDs: 2400

Accounts not found: 0
Call IDs not found in cleaned calls: 0
Agent IDs not found: 0


In [36]:
# Flag missing vendor IDs
attempts_clean["vendor_id_missing"] = (
    attempts_clean["vendor_id"].isna()
)

# Flag valid attempt numbers
attempts_clean["valid_attempt_no"] = (
    attempts_clean["attempt_no"].between(1, 7)
)

# Validate the dataset
print("Total records:", len(attempts_clean))

print("Duplicate attempt IDs:",
      attempts_clean["attempt_id"].duplicated().sum())

print("Missing vendor flags:",
      attempts_clean["vendor_id_missing"].sum())

print("Invalid attempt numbers:",
      (~attempts_clean["valid_attempt_no"]).sum())

print("Missing timestamps:",
      attempts_clean["event_at"].isna().sum())

Total records: 120000
Duplicate attempt IDs: 0
Missing vendor flags: 2400
Invalid attempt numbers: 0
Missing timestamps: 0


In [37]:
# Save cleaned call attempts dataset
output_file = os.path.join(
    cleaned_path, "call_attempts_cleaned.csv"
)

attempts_clean.to_csv(output_file, index=False)

print("Cleaned call attempts dataset saved!")
print(output_file)

# Verify saved file
attempts_check = pd.read_csv(output_file)

print("\nSaved dataset shape:", attempts_check.shape)
print("Duplicate attempt IDs:",
      attempts_check["attempt_id"].duplicated().sum())
print("Missing vendor flags:",
      attempts_check["vendor_id_missing"].sum())

Cleaned call attempts dataset saved!
/Users/simrannimje/Desktop/Collections_Analytics_Assignment/Data/Cleaned/call_attempts_cleaned.csv

Saved dataset shape: (120000, 11)
Duplicate attempt IDs: 0
Missing vendor flags: 2400


In [38]:
# Load call dispositions dataset
dispositions_raw = pd.read_csv(
    os.path.join(raw_path, "call_dispositions.csv")
)

print("Dataset shape:", dispositions_raw.shape)

print("\nData types:")
display(dispositions_raw.dtypes)

print("\nMissing values:")
display(dispositions_raw.isnull().sum())

print("\nFirst five records:")
display(dispositions_raw.head())

Dataset shape: (35000, 8)

Data types:


disposition_id         str
account_id             str
borrower_id            str
event_at               str
call_id                str
agent_id               str
disposition_code       str
disposition_version    str
dtype: object


Missing values:


disposition_id         0
account_id             0
borrower_id            0
event_at               0
call_id                0
agent_id               0
disposition_code       0
disposition_version    0
dtype: int64


First five records:


,disposition_id,account_id,borrower_id,event_at,call_id,agent_id,disposition_code,disposition_version
0,DISPOSITION0000001,ACC0029208,BRW0006282,2026-03-07 17:30:48,CALL0032272,AGT0000759,CALLBACK,v2
1,DISPOSITION0000002,ACC0009584,BRW0000540,2026-04-04 11:59:50,CALL0053880,AGT0000085,CALLBACK,legacy
2,DISPOSITION0000003,ACC0003386,BRW0010257,2026-03-11 01:25:13,CALL0048622,AGT0000319,PROMISE_TO_PAY,v1
3,DISPOSITION0000004,ACC0027899,BRW0006534,2026-03-17 14:49:43,CALL0072651,AGT0000189,NO_CONTACT,legacy
4,DISPOSITION0000005,ACC0024345,BRW0001918,2026-03-07 06:07:26,CALL0036866,AGT0000647,PTP,legacy


In [39]:
# Create a working copy
dispositions_clean = dispositions_raw.copy()

# Convert event timestamp
dispositions_clean["event_at"] = pd.to_datetime(
    dispositions_clean["event_at"], errors="coerce"
)

# Check duplicates
print("Exact duplicate rows:",
      dispositions_clean.duplicated().sum())

print("Duplicate disposition IDs:",
      dispositions_clean["disposition_id"].duplicated().sum())

# Check unique entities
print("\nUnique accounts:",
      dispositions_clean["account_id"].nunique())

print("Unique call IDs:",
      dispositions_clean["call_id"].nunique())

print("Unique agents:",
      dispositions_clean["agent_id"].nunique())

# Check disposition codes
print("\nDisposition codes:")
display(dispositions_clean["disposition_code"].value_counts())

# Check disposition versions
print("\nDisposition versions:")
display(dispositions_clean["disposition_version"].value_counts())

# Check call references
print("\nCall IDs not found:",
      (~dispositions_clean["call_id"].isin(
          calls_clean["call_id"]
      )).sum())

# Check account references
print("Accounts not found:",
      (~dispositions_clean["account_id"].isin(
          accounts_clean["account_id"]
      )).sum())

# Check agent references
print("Agents not found:",
      (~dispositions_clean["agent_id"].isin(
          agents_clean["agent_id"]
      )).sum())

# Check missing timestamps
print("Missing timestamps:",
      dispositions_clean["event_at"].isna().sum())

Exact duplicate rows: 0
Duplicate disposition IDs: 0

Unique accounts: 20603
Unique call IDs: 28971
Unique agents: 1000

Disposition codes:


disposition_code
NO_CONTACT        4010
WRONG_NUMBER      3946
PROMISE_TO_PAY    3926
PTP               3904
PTP_BROKEN        3900
CALLBACK          3872
DISPUTE           3826
PAID              3818
REFUSED           3798
Name: count, dtype: int64


Disposition versions:


disposition_version
legacy    11726
v2        11654
v1        11620
Name: count, dtype: int64


Call IDs not found: 0
Accounts not found: 0
Agents not found: 0
Missing timestamps: 0


In [40]:
# Preserve original disposition codes
dispositions_clean["disposition_code_original"] = (
    dispositions_clean["disposition_code"]
)

# Flag records using legacy coding
dispositions_clean["legacy_code_flag"] = (
    dispositions_clean["disposition_version"] == "legacy"
)

# Validate cleaned dataset
print("Total records:", len(dispositions_clean))

print("Duplicate disposition IDs:",
      dispositions_clean["disposition_id"].duplicated().sum())

print("Legacy code records:",
      dispositions_clean["legacy_code_flag"].sum())

print("Missing timestamps:",
      dispositions_clean["event_at"].isna().sum())

print("\nOriginal disposition codes preserved:")
display(dispositions_clean["disposition_code"].value_counts())

Total records: 35000
Duplicate disposition IDs: 0
Legacy code records: 11726
Missing timestamps: 0

Original disposition codes preserved:


disposition_code
NO_CONTACT        4010
WRONG_NUMBER      3946
PROMISE_TO_PAY    3926
PTP               3904
PTP_BROKEN        3900
CALLBACK          3872
DISPUTE           3826
PAID              3818
REFUSED           3798
Name: count, dtype: int64

In [41]:
# Save cleaned dispositions
dispositions_clean.to_csv(
    os.path.join(cleaned_path, "call_dispositions_cleaned.csv"),
    index=False
)

print("Saved successfully!")
print("Shape:", dispositions_clean.shape)

print("File:",
      os.path.join(cleaned_path, "call_dispositions_cleaned.csv"))

Saved successfully!
Shape: (35000, 10)
File: /Users/simrannimje/Desktop/Collections_Analytics_Assignment/Data/Cleaned/call_dispositions_cleaned.csv


In [42]:
# Load WhatsApp events
whatsapp_raw = pd.read_csv(
    os.path.join(raw_path, "whatsapp_events.csv")
)

# Inspect dataset
print("Dataset shape:", whatsapp_raw.shape)

print("\nData types:")
display(whatsapp_raw.dtypes)

print("\nMissing values:")
display(whatsapp_raw.isnull().sum())

print("\nFirst five records:")
display(whatsapp_raw.head())

Dataset shape: (60600, 8)

Data types:


whatsapp_event_id    str
account_id           str
borrower_id          str
event_at             str
message_id           str
event_type           str
template_code        str
provider_id          str
dtype: object


Missing values:


whatsapp_event_id    0
account_id           0
borrower_id          0
event_at             0
message_id           0
event_type           0
template_code        0
provider_id          0
dtype: int64


First five records:


,whatsapp_event_id,account_id,borrower_id,event_at,message_id,event_type,template_code,provider_id
0,WHATSAPP_EVENT0000001,ACC0006134,BRW0007859,2026-02-27 17:49:38,MSG000008171,FAILED,FIELD_VISIT,VND0000008
1,WHATSAPP_EVENT0000002,ACC0027479,BRW0003963,2026-02-09 11:55:18,MSG000000667,FAILED,PTP_01,VND0000003
2,WHATSAPP_EVENT0000003,ACC0008356,BRW0008340,2026-07-08 05:16:13,MSG000023608,DELIVERED,PAYMENT_LINK,VND0000001
3,WHATSAPP_EVENT0000004,ACC0024727,BRW0001572,2026-05-20 19:48:52,MSG000045438,DELIVERED,PAYMENT_LINK,VND0000010
4,WHATSAPP_EVENT0000005,ACC0024918,BRW0003130,2026-01-10 10:37:24,MSG000041041,READ,LEGAL_NOTICE,VND0000015


In [43]:
# Create a working copy
whatsapp_clean = whatsapp_raw.copy()

# Convert event timestamp
whatsapp_clean["event_at"] = pd.to_datetime(
    whatsapp_clean["event_at"],
    errors="coerce"
)

print("Exact duplicate rows:",
      whatsapp_clean.duplicated().sum())

print("Duplicate event IDs:",
      whatsapp_clean["whatsapp_event_id"].duplicated().sum())

print("Unique message IDs:",
      whatsapp_clean["message_id"].nunique())

print("Unique accounts:",
      whatsapp_clean["account_id"].nunique())

print("Unique borrowers:",
      whatsapp_clean["borrower_id"].nunique())

print("\nEvent type distribution:")
display(whatsapp_clean["event_type"].value_counts())

print("\nTemplate distribution:")
display(whatsapp_clean["template_code"].value_counts())

print("\nProvider distribution:")
display(whatsapp_clean["provider_id"].value_counts())

Exact duplicate rows: 600
Duplicate event IDs: 600
Unique message IDs: 34831
Unique accounts: 25924
Unique borrowers: 11917

Event type distribution:


event_type
SENT             10244
PAYMENT_CLICK    10180
FAILED           10114
REPLIED          10035
READ             10022
DELIVERED        10005
Name: count, dtype: int64


Template distribution:


template_code
PTP_01          12275
FIELD_VISIT     12177
PAYMENT_LINK    12152
LEGAL_NOTICE    12070
REMINDER_01     11926
Name: count, dtype: int64


Provider distribution:


provider_id
VND0000002    4177
VND0000007    4154
VND0000015    4094
VND0000005    4089
VND0000013    4085
VND0000008    4053
VND0000011    4052
VND0000014    4049
VND0000001    4043
VND0000012    4022
VND0000010    3981
VND0000006    3980
VND0000004    3970
VND0000009    3961
VND0000003    3890
Name: count, dtype: int64

In [44]:
# Analyse message IDs mapped to multiple accounts and borrowers

message_check = whatsapp_clean.groupby("message_id").agg(
    account_count=("account_id", "nunique"),
    borrower_count=("borrower_id", "nunique"),
    template_count=("template_code", "nunique"),
    provider_count=("provider_id", "nunique"),
    event_count=("whatsapp_event_id", "count")
)

print("Message IDs:", len(message_check))

print("Messages linked to multiple accounts:",
      (message_check["account_count"] > 1).sum())

print("Messages linked to multiple borrowers:",
      (message_check["borrower_count"] > 1).sum())

print("Messages linked to multiple templates:",
      (message_check["template_count"] > 1).sum())

print("Messages linked to multiple providers:",
      (message_check["provider_count"] > 1).sum())

print("\nMessage event count distribution:")
display(message_check["event_count"].value_counts().sort_index())

Message IDs: 34831
Messages linked to multiple accounts: 16818
Messages linked to multiple borrowers: 16818
Messages linked to multiple templates: 14517
Messages linked to multiple providers: 16042

Message event count distribution:


event_count
1    17832
2    10679
3     4447
4     1422
5      348
6       85
7       14
8        3
9        1
Name: count, dtype: int64

In [45]:
# Remove exact duplicate event records
whatsapp_clean = whatsapp_clean.drop_duplicates().copy()

# Flag repeated event IDs
whatsapp_clean["duplicate_event_id_flag"] = (
    whatsapp_clean["whatsapp_event_id"].duplicated(keep=False)
)

# Validate cleaned dataset
print("Records after removing exact duplicates:", len(whatsapp_clean))

print("Remaining duplicate event IDs:",
      whatsapp_clean["whatsapp_event_id"].duplicated().sum())

print("Missing timestamps:",
      whatsapp_clean["event_at"].isna().sum())

# Save cleaned dataset
whatsapp_clean.to_csv(
    os.path.join(cleaned_path, "whatsapp_events_cleaned.csv"),
    index=False
)

print("\nWhatsApp dataset saved successfully!")

Records after removing exact duplicates: 60000
Remaining duplicate event IDs: 0
Missing timestamps: 0

WhatsApp dataset saved successfully!


In [46]:
# Load SMS events
sms_raw = pd.read_csv(
    os.path.join(raw_path, "sms_events.csv")
)

# Create working copy
sms_clean = sms_raw.copy()

# Convert event timestamp
sms_clean["event_at"] = pd.to_datetime(
    sms_clean["event_at"],
    errors="coerce"
)

print("Dataset shape:", sms_clean.shape)

print("\nData types:")
display(sms_clean.dtypes)

print("\nMissing values:")
display(sms_clean.isnull().sum())

print("\nFirst five records:")
display(sms_clean.head())

Dataset shape: (45000, 8)

Data types:


sms_event_id                str
account_id                  str
borrower_id                 str
event_at         datetime64[us]
message_id                  str
event_type                  str
template_code               str
provider_id                 str
dtype: object


Missing values:


sms_event_id     0
account_id       0
borrower_id      0
event_at         0
message_id       0
event_type       0
template_code    0
provider_id      0
dtype: int64


First five records:


,sms_event_id,account_id,borrower_id,event_at,message_id,event_type,template_code,provider_id
0,SMS_EVENT0000001,ACC0010751,BRW0009574,2026-04-24 20:57:06,SMS000022764,CLICKED,PAYMENT_LINK,VND0000010
1,SMS_EVENT0000002,ACC0005842,BRW0003619,2026-05-24 11:28:20,SMS000015050,DELIVERED,NOTICE,VND0000003
2,SMS_EVENT0000003,ACC0014253,BRW0003018,2026-03-05 20:23:20,SMS000028337,CLICKED,NOTICE,VND0000008
3,SMS_EVENT0000004,ACC0016374,BRW0002587,2026-05-11 04:56:15,SMS000004875,DELIVERED,DUE_REMINDER,VND0000010
4,SMS_EVENT0000005,ACC0000796,BRW0010499,2026-03-24 08:41:53,SMS000008899,SENT,PTP_REMINDER,VND0000007


In [47]:
# Check duplicate records and IDs
print("Exact duplicate rows:", sms_clean.duplicated().sum())

print("Duplicate SMS event IDs:",
      sms_clean["sms_event_id"].duplicated().sum())

print("Unique message IDs:", sms_clean["message_id"].nunique())

print("Unique accounts:", sms_clean["account_id"].nunique())

print("Unique borrowers:", sms_clean["borrower_id"].nunique())

print("\nEvent type distribution:")
display(sms_clean["event_type"].value_counts())

print("\nTemplate distribution:")
display(sms_clean["template_code"].value_counts())

print("\nProvider distribution:")
display(sms_clean["provider_id"].value_counts())

Exact duplicate rows: 0
Duplicate SMS event IDs: 0
Unique message IDs: 27001
Unique accounts: 23207
Unique borrowers: 11728

Event type distribution:


event_type
SENT         11391
FAILED       11251
DELIVERED    11219
CLICKED      11139
Name: count, dtype: int64


Template distribution:


template_code
PAYMENT_LINK    11420
DUE_REMINDER    11256
NOTICE          11166
PTP_REMINDER    11158
Name: count, dtype: int64


Provider distribution:


provider_id
VND0000012    3087
VND0000010    3068
VND0000015    3064
VND0000001    3013
VND0000008    3009
VND0000014    3003
VND0000011    3001
VND0000007    2992
VND0000006    2989
VND0000004    2988
VND0000005    2985
VND0000013    2973
VND0000003    2971
VND0000002    2935
VND0000009    2922
Name: count, dtype: int64

In [48]:
# Check whether message IDs map consistently

sms_message_check = sms_clean.groupby("message_id").agg(
    account_count=("account_id", "nunique"),
    borrower_count=("borrower_id", "nunique"),
    template_count=("template_code", "nunique"),
    provider_count=("provider_id", "nunique"),
    event_count=("sms_event_id", "count")
)

print("Message IDs:", len(sms_message_check))

print("Messages linked to multiple accounts:",
      (sms_message_check["account_count"] > 1).sum())

print("Messages linked to multiple borrowers:",
      (sms_message_check["borrower_count"] > 1).sum())

print("Messages linked to multiple templates:",
      (sms_message_check["template_count"] > 1).sum())

print("Messages linked to multiple providers:",
      (sms_message_check["provider_count"] > 1).sum())

print("\nMessage event count distribution:")
display(
    sms_message_check["event_count"].value_counts().sort_index()
)

Message IDs: 27001
Messages linked to multiple accounts: 12354
Messages linked to multiple borrowers: 12354
Messages linked to multiple templates: 10069
Messages linked to multiple providers: 11781

Message event count distribution:


event_count
1    14647
2     8106
3     3162
4      834
5      202
6       42
7        7
8        1
Name: count, dtype: int64

In [49]:
# Flag message IDs linked to multiple accounts
sms_clean["message_mapping_issue_flag"] = (
    sms_clean["message_id"].map(
        sms_message_check["account_count"]
    ) > 1
)

# Validate cleaned dataset
print("Final records:", len(sms_clean))

print("Duplicate SMS event IDs:",
      sms_clean["sms_event_id"].duplicated().sum())

print("Messages with mapping issues:",
      sms_clean["message_mapping_issue_flag"].sum())

# Save cleaned dataset
sms_clean.to_csv(
    os.path.join(cleaned_path, "sms_events_cleaned.csv"),
    index=False
)

print("\nSMS dataset saved successfully!")

Final records: 45000
Duplicate SMS event IDs: 0
Messages with mapping issues: 30353

SMS dataset saved successfully!


In [50]:
# Load field visits
visits_raw = pd.read_csv(
    os.path.join(raw_path, "field_visits.csv")
)

# Create working copy
visits_clean = visits_raw.copy()

# Convert timestamps
visits_clean["event_at"] = pd.to_datetime(
    visits_clean["event_at"],
    errors="coerce"
)

visits_clean["scheduled_at"] = pd.to_datetime(
    visits_clean["scheduled_at"],
    errors="coerce"
)

print("Dataset shape:", visits_clean.shape)

print("\nData types:")
display(visits_clean.dtypes)

print("\nMissing values:")
display(visits_clean.isnull().sum())

print("\nFirst five records:")
display(visits_clean.head())

Dataset shape: (25000, 10)

Data types:


visit_id                   str
account_id                 str
borrower_id                str
event_at        datetime64[us]
agent_id                   str
visit_type                 str
outcome                    str
latitude               float64
longitude              float64
scheduled_at    datetime64[us]
dtype: object


Missing values:


visit_id          0
account_id        0
borrower_id       0
event_at          0
agent_id          0
visit_type        0
outcome           0
latitude          0
longitude         0
scheduled_at    250
dtype: int64


First five records:


,visit_id,account_id,borrower_id,event_at,agent_id,visit_type,outcome,latitude,longitude,scheduled_at
0,VISIT0000001,ACC0026237,BRW0009469,2026-01-25 01:15:48,AGT0000403,AFTER_HOURS,PAID,31.861706,76.098184,2026-01-24 05:49:28
1,VISIT0000002,ACC0026942,BRW0010461,2026-06-23 22:18:25,AGT0000278,NOTICE,PAID,23.641189,74.571935,2026-06-23 14:17:54
2,VISIT0000003,ACC0022809,BRW0006318,2026-05-16 08:06:34,AGT0000723,NOTICE,CONTACTED,22.324370,84.790537,2026-05-16 02:58:39
3,VISIT0000004,ACC0029193,BRW0005195,2026-06-14 00:12:17,AGT0000946,FOLLOWUP,CONTACTED,27.230510,75.957354,2026-06-13 13:32:15
4,VISIT0000005,ACC0015605,BRW0009229,2026-08-01 21:31:09,AGT0000917,NOTICE,REFUSED,23.563588,85.547376,2026-08-01 00:47:29


In [51]:
# Check duplicate records and visit IDs
print("Exact duplicate rows:", visits_clean.duplicated().sum())

print("Duplicate visit IDs:",
      visits_clean["visit_id"].duplicated().sum())

print("Unique accounts:", visits_clean["account_id"].nunique())

print("Unique agents:", visits_clean["agent_id"].nunique())

print("\nVisit type distribution:")
display(visits_clean["visit_type"].value_counts())

print("\nVisit outcome distribution:")
display(visits_clean["outcome"].value_counts())

print("\nMissing scheduled timestamps:",
      visits_clean["scheduled_at"].isna().sum())

# Check timestamp sequence
print("\nVisits occurring before scheduled time:",
      (visits_clean["event_at"] < visits_clean["scheduled_at"]).sum())

Exact duplicate rows: 0
Duplicate visit IDs: 0
Unique accounts: 16908
Unique agents: 1000

Visit type distribution:


visit_type
FIRST_VISIT    6275
FOLLOWUP       6271
NOTICE         6262
AFTER_HOURS    6192
Name: count, dtype: int64


Visit outcome distribution:


outcome
WRONG_ADDRESS    4221
CONTACTED        4205
PTP              4200
NOT_AVAILABLE    4190
REFUSED          4104
PAID             4080
Name: count, dtype: int64


Missing scheduled timestamps: 250

Visits occurring before scheduled time: 0


In [52]:
# Flag missing scheduled timestamps
visits_clean["scheduled_at_missing_flag"] = (
    visits_clean["scheduled_at"].isna()
)

# Save cleaned dataset
visits_clean.to_csv(
    os.path.join(cleaned_path, "field_visits_cleaned.csv"),
    index=False
)

print("Final records:", len(visits_clean))

print("Missing scheduled timestamps:",
      visits_clean["scheduled_at_missing_flag"].sum())

print("\nField Visits dataset saved successfully!")

Final records: 25000
Missing scheduled timestamps: 250

Field Visits dataset saved successfully!


In [53]:
# Load complaints
complaints_raw = pd.read_csv(
    os.path.join(raw_path, "complaints.csv")
)

# Create working copy
complaints_clean = complaints_raw.copy()

# Convert timestamps
complaints_clean["event_at"] = pd.to_datetime(
    complaints_clean["event_at"],
    errors="coerce"
)

complaints_clean["resolution_at"] = pd.to_datetime(
    complaints_clean["resolution_at"],
    errors="coerce"
)

print("Dataset shape:", complaints_clean.shape)

print("\nData types:")
display(complaints_clean.dtypes)

print("\nMissing values:")
display(complaints_clean.isnull().sum())

print("\nFirst five records:")
display(complaints_clean.head())

Dataset shape: (8000, 9)

Data types:


complaint_id                 str
account_id                   str
borrower_id                  str
event_at          datetime64[us]
complaint_type               str
severity                     str
status                       str
source                       str
resolution_at     datetime64[us]
dtype: object


Missing values:


complaint_id      0
account_id        0
borrower_id       0
event_at          0
complaint_type    0
severity          0
status            0
source            0
resolution_at     0
dtype: int64


First five records:


,complaint_id,account_id,borrower_id,event_at,complaint_type,severity,status,source,resolution_at
0,COMPLAINT0000001,ACC0015256,BRW0010194,2026-04-29 16:34:08,DISPUTE,HIGH,CLOSED,FIELD,2026-05-12 16:34:08
1,COMPLAINT0000002,ACC0002310,BRW0009657,2026-06-20 04:52:15,HARASSMENT,MEDIUM,RESOLVED,CALL,2026-06-28 04:52:15
2,COMPLAINT0000003,ACC0002989,BRW0008434,2026-01-29 14:31:42,WRONG_CONTACT,MEDIUM,INVESTIGATING,WHATSAPP,2026-02-07 14:31:42
3,COMPLAINT0000004,ACC0005034,BRW0011197,2026-07-06 03:59:49,DISPUTE,HIGH,INVESTIGATING,EMAIL,2026-07-08 03:59:49
4,COMPLAINT0000005,ACC0017220,BRW0001397,2026-05-30 18:10:30,DND,MEDIUM,RESOLVED,FIELD,2026-06-09 18:10:30


In [54]:
# Check duplicate records and complaint IDs
print("Exact duplicate rows:", complaints_clean.duplicated().sum())

print("Duplicate complaint IDs:",
      complaints_clean["complaint_id"].duplicated().sum())

print("Unique accounts:", complaints_clean["account_id"].nunique())

print("Unique borrowers:", complaints_clean["borrower_id"].nunique())

print("\nComplaint type distribution:")
display(complaints_clean["complaint_type"].value_counts())

print("\nSeverity distribution:")
display(complaints_clean["severity"].value_counts())

print("\nComplaint status distribution:")
display(complaints_clean["status"].value_counts())

print("\nComplaint source distribution:")
display(complaints_clean["source"].value_counts())

# Check timestamp consistency
print("\nResolution timestamps before complaint dates:",
      (complaints_clean["resolution_at"] <
       complaints_clean["event_at"]).sum())

print("Open complaints with resolution timestamps:",
      ((complaints_clean["status"] == "OPEN") &
       complaints_clean["resolution_at"].notna()).sum())

Exact duplicate rows: 0
Duplicate complaint IDs: 0
Unique accounts: 7034
Unique borrowers: 5839

Complaint type distribution:


complaint_type
AGENT_BEHAVIOUR    1202
DISPUTE            1155
PRIVACY            1155
DND                1152
HARASSMENT         1126
PAYMENT            1112
WRONG_CONTACT      1098
Name: count, dtype: int64


Severity distribution:


severity
MEDIUM      2047
HIGH        1997
LOW         1988
CRITICAL    1968
Name: count, dtype: int64


Complaint status distribution:


status
OPEN             2096
INVESTIGATING    2005
CLOSED           1964
RESOLVED         1935
Name: count, dtype: int64


Complaint source distribution:


source
FIELD       1647
EMAIL       1607
WHATSAPP    1589
SMS         1579
CALL        1578
Name: count, dtype: int64


Resolution timestamps before complaint dates: 0
Open complaints with resolution timestamps: 2096


In [55]:
# Flag open complaints with resolution timestamps
complaints_clean["open_with_resolution_flag"] = (
    (complaints_clean["status"] == "OPEN") &
    (complaints_clean["resolution_at"].notna())
)

# Flag high-severity complaints
complaints_clean["high_severity_flag"] = (
    complaints_clean["severity"].isin(["HIGH", "CRITICAL"])
)

# Save cleaned dataset
complaints_clean.to_csv(
    os.path.join(cleaned_path, "complaints_cleaned.csv"),
    index=False
)

print("Final records:", len(complaints_clean))

print("Open complaints with resolution timestamps:",
      complaints_clean["open_with_resolution_flag"].sum())

print("High/Critical complaints:",
      complaints_clean["high_severity_flag"].sum())

print("\nComplaints dataset saved successfully!")

Final records: 8000
Open complaints with resolution timestamps: 2096
High/Critical complaints: 3965

Complaints dataset saved successfully!


In [56]:
# Load account status history
history_raw = pd.read_csv(
    os.path.join(raw_path, "account_status_history.csv")
)

# Create working copy
history_clean = history_raw.copy()

# Convert timestamps
history_clean["event_at"] = pd.to_datetime(
    history_clean["event_at"],
    errors="coerce"
)

history_clean["recorded_at"] = pd.to_datetime(
    history_clean["recorded_at"],
    errors="coerce"
)

print("Dataset shape:", history_clean.shape)

print("\nData types:")
display(history_clean.dtypes)

print("\nMissing values:")
display(history_clean.isnull().sum())

print("\nFirst five records:")
display(history_clean.head())

Dataset shape: (60000, 8)

Data types:


history_id                str
account_id                str
borrower_id               str
event_at       datetime64[us]
status                    str
changed_by                str
source                    str
recorded_at    datetime64[us]
dtype: object


Missing values:


history_id     0
account_id     0
borrower_id    0
event_at       0
status         0
changed_by     0
source         0
recorded_at    0
dtype: int64


First five records:


,history_id,account_id,borrower_id,event_at,status,changed_by,source,recorded_at
0,HISTORY0000001,ACC0004322,BRW0005490,2026-07-03 04:28:38,WRITEOFF,AGT0000039,CALL,2026-07-03 00:41:54
1,HISTORY0000002,ACC0024644,BRW0005343,2026-07-18 02:03:23,DELINQUENT,AGT0000059,CALL,2026-07-18 22:42:02
2,HISTORY0000003,ACC0020233,BRW0003150,2026-04-15 09:17:47,CLOSED,AGT0000080,PAYMENT,2026-04-15 19:40:51
3,HISTORY0000004,ACC0004384,BRW0008483,2026-01-16 10:52:04,WRITEOFF,SYSTEM,CORE,2026-01-16 23:05:28
4,HISTORY0000005,ACC0022967,BRW0009477,2026-05-22 04:13:04,PTP,AGT0000073,CORE,2026-05-22 07:08:28


In [57]:
# Check duplicate records and history IDs
print("Exact duplicate rows:", history_clean.duplicated().sum())

print("Duplicate history IDs:",
      history_clean["history_id"].duplicated().sum())

print("Unique accounts:", history_clean["account_id"].nunique())

print("Unique borrowers:", history_clean["borrower_id"].nunique())

print("\nStatus distribution:")
display(history_clean["status"].value_counts())

print("\nSource distribution:")
display(history_clean["source"].value_counts())

# Check recording delay
history_clean["recording_delay_hours"] = (
    history_clean["recorded_at"] - history_clean["event_at"]
).dt.total_seconds() / 3600

print("\nRecording delay summary (hours):")
display(history_clean["recording_delay_hours"].describe())

print("\nRecords recorded before the event timestamp:",
      (history_clean["recording_delay_hours"] < 0).sum())

print("\nRecords with delay greater than 24 hours:",
      (history_clean["recording_delay_hours"] > 24).sum())

Exact duplicate rows: 0
Duplicate history IDs: 0
Unique accounts: 25999
Unique borrowers: 11916

Status distribution:


status
PAID          8650
CLOSED        8614
DELINQUENT    8612
NPA           8612
WRITEOFF      8583
ACTIVE        8518
PTP           8411
Name: count, dtype: int64


Source distribution:


source
CORE       12079
BATCH      12073
CALL       12028
FIELD      11911
PAYMENT    11909
Name: count, dtype: int64


Recording delay summary (hours):


count    60000.000000
mean        -0.096023
std         13.873956
min        -23.998611
25%        -12.128403
50%         -0.125972
75%         11.926111
max         23.999444
Name: recording_delay_hours, dtype: float64


Records recorded before the event timestamp: 30191

Records with delay greater than 24 hours: 0


In [58]:
# Sort status history chronologically
history_sorted = history_clean.sort_values(
    ["account_id", "event_at"]
).copy()

# Identify previous status for each account
history_sorted["previous_status"] = (
    history_sorted.groupby("account_id")["status"].shift(1)
)

# Identify status transitions
history_sorted["status_transition"] = (
    history_sorted["previous_status"].fillna("START")
    + " -> "
    + history_sorted["status"]
)

print("Total status records:", len(history_sorted))

print("Accounts with multiple status records:",
      (history_sorted.groupby("account_id").size() > 1).sum())

print("\nMost frequent status transitions:")
display(
    history_sorted["status_transition"]
    .value_counts()
    .head(20)
)

print("\nSelected potentially unusual transitions:")
display(
    history_sorted[
        history_sorted["status_transition"].isin([
            "PAID -> DELINQUENT",
            "CLOSED -> PAID",
            "WRITEOFF -> DELINQUENT"
        ])
    ][["account_id", "event_at", "previous_status", "status"]].head(15)
)

Total status records: 60000
Accounts with multiple status records: 17821

Most frequent status transitions:


status_transition
START -> CLOSED             3800
START -> NPA                3745
START -> ACTIVE             3733
START -> PAID               3725
START -> WRITEOFF           3718
START -> DELINQUENT         3717
START -> PTP                3561
NPA -> PAID                  735
CLOSED -> PAID               732
DELINQUENT -> CLOSED         729
WRITEOFF -> DELINQUENT       729
CLOSED -> DELINQUENT         725
DELINQUENT -> NPA            722
PTP -> PTP                   722
DELINQUENT -> DELINQUENT     720
CLOSED -> WRITEOFF           718
DELINQUENT -> PAID           718
WRITEOFF -> PAID             716
NPA -> PTP                   713
ACTIVE -> CLOSED             711
Name: count, dtype: int64


Selected potentially unusual transitions:


,account_id,event_at,previous_status,status
29227,ACC0000011,2026-06-13 15:38:45,CLOSED,PAID
33672,ACC0000044,2026-06-19 04:29:04,CLOSED,PAID
15082,ACC0000052,2026-07-21 13:39:57,WRITEOFF,DELINQUENT
19089,ACC0000109,2026-06-28 06:01:50,CLOSED,PAID
44830,ACC0000132,2026-03-27 21:18:16,PAID,DELINQUENT
56795,ACC0000138,2026-06-10 18:24:30,CLOSED,PAID
56080,ACC0000168,2026-06-28 02:36:00,WRITEOFF,DELINQUENT
51251,ACC0000181,2026-06-23 08:13:23,CLOSED,PAID
38533,ACC0000188,2026-06-22 20:52:20,CLOSED,PAID
8083,ACC0000194,2026-07-23 13:58:32,WRITEOFF,DELINQUENT


In [59]:
# Calculate time between consecutive status events
history_sorted["previous_event_at"] = (
    history_sorted.groupby("account_id")["event_at"].shift(1)
)

history_sorted["hours_since_previous_status"] = (
    history_sorted["event_at"] -
    history_sorted["previous_event_at"]
).dt.total_seconds() / 3600

# Flag selected unusual transitions
unusual_transitions = [
    "PAID -> DELINQUENT",
    "CLOSED -> PAID",
    "WRITEOFF -> DELINQUENT"
]

history_sorted["unusual_transition_flag"] = (
    history_sorted["status_transition"].isin(unusual_transitions)
)

# Flag recording timestamps earlier than event timestamps
history_sorted["recording_before_event_flag"] = (
    history_sorted["recorded_at"] < history_sorted["event_at"]
)

print("Unusual transition records:",
      history_sorted["unusual_transition_flag"].sum())

print("Recording-before-event records:",
      history_sorted["recording_before_event_flag"].sum())

print("\nTime between consecutive status events:")
display(
    history_sorted["hours_since_previous_status"].describe()
)

Unusual transition records: 2168
Recording-before-event records: 30191

Time between consecutive status events:


count    34001.000000
mean      1257.431617
std       1044.373088
min          0.003333
25%        423.792222
50%        984.140278
75%       1847.204722
max       5217.371944
Name: hours_since_previous_status, dtype: float64

In [60]:
# Add data-quality flags
history_sorted["recording_before_event_flag"] = (
    history_sorted["recorded_at"] < history_sorted["event_at"]
)

history_sorted["unusual_transition_flag"] = (
    history_sorted["status_transition"].isin([
        "PAID -> DELINQUENT",
        "CLOSED -> PAID",
        "WRITEOFF -> DELINQUENT"
    ])
)

# Save cleaned dataset
history_sorted.to_csv(
    os.path.join(cleaned_path, "account_status_history_cleaned.csv"),
    index=False
)

print("Final records:", len(history_sorted))

print("Unusual transitions:",
      history_sorted["unusual_transition_flag"].sum())

print("Recording-before-event flags:",
      history_sorted["recording_before_event_flag"].sum())

print("\nAccount Status History saved successfully!")

Final records: 60000
Unusual transitions: 2168
Recording-before-event flags: 30191

Account Status History saved successfully!


In [61]:
# Load accounts
accounts_raw = pd.read_csv(
    os.path.join(raw_path, "accounts.csv")
)

# Create working copy
accounts_clean = accounts_raw.copy()

# Convert numeric columns
accounts_clean["principal_amount"] = pd.to_numeric(
    accounts_clean["principal_amount"], errors="coerce"
)

accounts_clean["outstanding_amount"] = pd.to_numeric(
    accounts_clean["outstanding_amount"], errors="coerce"
)

accounts_clean["dpd"] = pd.to_numeric(
    accounts_clean["dpd"], errors="coerce"
)

# Convert account opening timestamp
accounts_clean["opened_at"] = pd.to_datetime(
    accounts_clean["opened_at"], errors="coerce"
)

print("Dataset shape:", accounts_clean.shape)

print("\nData types:")
display(accounts_clean.dtypes)

print("\nMissing values:")
display(accounts_clean.isnull().sum())

print("\nFirst five records:")
display(accounts_clean.head())

Dataset shape: (30000, 11)

Data types:


account_id                       str
borrower_id                      str
loan_type                        str
principal_amount             float64
outstanding_amount           float64
dpd                            int64
risk_segment                     str
status                           str
opened_at             datetime64[us]
timezone                         str
schema_version                   str
dtype: object


Missing values:


account_id              0
borrower_id           455
loan_type               0
principal_amount        0
outstanding_amount      0
dpd                     0
risk_segment            0
status                  0
opened_at               0
timezone                0
schema_version          0
dtype: int64


First five records:


,account_id,borrower_id,loan_type,principal_amount,outstanding_amount,dpd,risk_segment,status,opened_at,timezone,schema_version
0,ACC0000001,BRW0010742,CONSUMER,603443.40,678074.03,15,MEDIUM,CLOSED,2025-11-11 04:37:00,Asia/Dubai,v1
1,ACC0000002,BRW0009382,BNPL,277190.14,464893.44,5,LOW,ACTIVE,2025-11-13 15:59:44,UTC,v3
2,ACC0000003,BRW0003966,CREDIT_CARD,628658.43,22565.37,60,NPA,WRITEOFF,2025-09-12 04:59:20,Asia/Dubai,v2
3,ACC0000004,BRW0001993,PERSONAL,76435.07,508427.73,30,LOW,CLOSED,2025-05-25 19:29:38,UTC,v2
4,ACC0000005,BRW0009976,CONSUMER,646797.08,563858.01,180,LOW,WRITEOFF,2025-09-07 14:59:13,Asia/Dubai,v2


In [62]:
# Check duplicate account IDs
print("Duplicate account IDs:",
      accounts_clean["account_id"].duplicated().sum())

# Check outstanding amount against principal
accounts_clean["outstanding_above_principal_flag"] = (
    accounts_clean["outstanding_amount"] >
    accounts_clean["principal_amount"]
)

print("Outstanding above principal:",
      accounts_clean["outstanding_above_principal_flag"].sum())

# Check invalid numeric values
print("\nZero or negative principal:",
      (accounts_clean["principal_amount"] <= 0).sum())

print("Zero or negative outstanding:",
      (accounts_clean["outstanding_amount"] <= 0).sum())

print("Negative DPD:",
      (accounts_clean["dpd"] < 0).sum())

# Check distributions
print("\nLoan type distribution:")
display(accounts_clean["loan_type"].value_counts())

print("\nRisk segment distribution:")
display(accounts_clean["risk_segment"].value_counts())

print("\nDPD distribution:")
display(accounts_clean["dpd"].value_counts().sort_index())

print("\nTimezone distribution:")
display(accounts_clean["timezone"].value_counts())

print("\nSchema version distribution:")
display(accounts_clean["schema_version"].value_counts())

Duplicate account IDs: 0
Outstanding above principal: 13029

Zero or negative principal: 0
Zero or negative outstanding: 0
Negative DPD: 0

Loan type distribution:


loan_type
CREDIT_CARD    6080
AUTO           6079
PERSONAL       5983
CONSUMER       5930
BNPL           5928
Name: count, dtype: int64


Risk segment distribution:


risk_segment
HIGH      7552
MEDIUM    7533
LOW       7513
NPA       7402
Name: count, dtype: int64


DPD distribution:


dpd
0      2685
1      2713
5      2727
15     2736
30     2704
45     2744
60     2770
75     2741
90     2727
120    2759
180    2694
Name: count, dtype: int64


Timezone distribution:


timezone
UTC             10096
Asia/Kolkata     9981
Asia/Dubai       9923
Name: count, dtype: int64


Schema version distribution:


schema_version
v1    10152
v2    10026
v3     9822
Name: count, dtype: int64

In [63]:
# Flag missing borrower IDs
accounts_clean["borrower_id_missing_flag"] = (
    accounts_clean["borrower_id"].isna()
)

# Save cleaned dataset
accounts_clean.to_csv(
    os.path.join(cleaned_path, "accounts_cleaned.csv"),
    index=False
)

print("Final records:", len(accounts_clean))

print("Missing borrower IDs:",
      accounts_clean["borrower_id_missing_flag"].sum())

print("Outstanding above principal:",
      accounts_clean["outstanding_above_principal_flag"].sum())

print("\nAccounts dataset saved successfully!")

Final records: 30000
Missing borrower IDs: 455
Outstanding above principal: 13029

Accounts dataset saved successfully!


In [64]:
# Load vendor telephony data
vendors_raw = pd.read_csv(
    os.path.join(raw_path, "vendor_telephony.csv")
)

# Create working copy
vendors_clean = vendors_raw.copy()

print("Dataset shape:", vendors_clean.shape)

print("\nData types:")
display(vendors_clean.dtypes)

print("\nMissing values:")
display(vendors_clean.isnull().sum())

print("\nFirst five records:")
display(vendors_clean.head())

Dataset shape: (15, 6)

Data types:


vendor_id            str
vendor_name          str
vendor_account_id    str
timezone             str
status               str
schema_version       str
dtype: object


Missing values:


vendor_id            0
vendor_name          0
vendor_account_id    0
timezone             0
status               0
schema_version       0
dtype: int64


First five records:


,vendor_id,vendor_name,vendor_account_id,timezone,status,schema_version
0,VND0000001,Airtel,VAC342762,Asia/Kolkata,INACTIVE,v3
1,VND0000002,Exotel,VAC456766,UTC,INACTIVE,v3
2,VND0000003,Twilio,VAC074211,UTC,INACTIVE,v3
3,VND0000004,Twilio,VAC321507,UTC,ACTIVE,v1
4,VND0000005,Twilio,VAC976733,Asia/Kolkata,ACTIVE,v1


In [65]:
# Check duplicate vendor IDs and account IDs
print("Duplicate vendor IDs:",
      vendors_clean["vendor_id"].duplicated().sum())

print("Duplicate vendor account IDs:",
      vendors_clean["vendor_account_id"].duplicated().sum())

print("\nVendor names:")
display(vendors_clean["vendor_name"].value_counts())

print("\nVendor status:")
display(vendors_clean["status"].value_counts())

print("\nTimezone distribution:")
display(vendors_clean["timezone"].value_counts())

print("\nSchema version distribution:")
display(vendors_clean["schema_version"].value_counts())

# Check vendor IDs referenced in call attempts
print("\nVendor IDs in call attempts:",
      attempts_clean["vendor_id"].nunique())

print("Missing vendor IDs in call attempts:",
      attempts_clean["vendor_id"].isna().sum())

Duplicate vendor IDs: 0
Duplicate vendor account IDs: 0

Vendor names:


vendor_name
Airtel        5
Twilio        3
Knowlarity    3
Exotel        2
TataTele      2
Name: count, dtype: int64


Vendor status:


status
INACTIVE    9
ACTIVE      6
Name: count, dtype: int64


Timezone distribution:


timezone
UTC             8
Asia/Kolkata    7
Name: count, dtype: int64


Schema version distribution:


schema_version
v1    7
v3    6
v2    2
Name: count, dtype: int64


Vendor IDs in call attempts: 15
Missing vendor IDs in call attempts: 2400


In [66]:
# Check vendor status against call attempts
vendor_attempt_check = attempts_clean.merge(
    vendors_clean[["vendor_id", "vendor_name", "status"]],
    on="vendor_id",
    how="left"
)

print("Call attempts by vendor status:")
display(vendor_attempt_check["status"].value_counts(dropna=False))

print("\nCall attempts by vendor:")
display(
    vendor_attempt_check.groupby(
        ["vendor_id", "vendor_name", "status"],
        dropna=False
    ).size().reset_index(name="attempt_count")
)

# Check vendor reference integrity
print("\nUnmatched vendor IDs:",
      vendor_attempt_check["vendor_name"].isna().sum())

Call attempts by vendor status:


status
INACTIVE    70500
ACTIVE      47100
NaN          2400
Name: count, dtype: int64


Call attempts by vendor:


,vendor_id,vendor_name,status,attempt_count
0,VND0000001,Airtel,INACTIVE,7962
1,VND0000002,Exotel,INACTIVE,7715
2,VND0000003,Twilio,INACTIVE,7851
3,VND0000004,Twilio,ACTIVE,7920
4,VND0000005,Twilio,ACTIVE,7748
5,VND0000006,Knowlarity,ACTIVE,7932
6,VND0000007,TataTele,INACTIVE,7670
7,VND0000008,Airtel,INACTIVE,7810
8,VND0000009,Exotel,INACTIVE,7788
9,VND0000010,Airtel,INACTIVE,7822



Unmatched vendor IDs: 2400


In [67]:
# Add vendor status quality flags
vendors_clean["vendor_inactive_flag"] = (
    vendors_clean["status"] == "INACTIVE"
)

# Save cleaned vendor master
vendors_clean.to_csv(
    os.path.join(cleaned_path, "vendor_telephony_cleaned.csv"),
    index=False
)

# Final validation
print("Total vendors:", len(vendors_clean))

print("Duplicate vendor IDs:",
      vendors_clean["vendor_id"].duplicated().sum())

print("\nVendor dataset saved successfully!")

Total vendors: 15
Duplicate vendor IDs: 0

Vendor dataset saved successfully!


In [68]:
import os

raw_path = os.path.join(project_path, "Data", "Raw")

print("Files available in Raw folder:\n")

for file in os.listdir(raw_path):
    if "ptp" in file.lower() or "promise" in file.lower():
        print(file)

Files available in Raw folder:

promises_to_pay.csv


In [69]:
import pandas as pd
import numpy as np
import os

project_path = "/Users/simrannimje/Desktop/Collections_Analytics_Assignment"

raw_path = os.path.join(project_path, "Data", "Raw")
cleaned_path = os.path.join(project_path, "Data", "Cleaned")

# Load raw PTP data
ptp_raw = pd.read_csv(
    os.path.join(raw_path, "promises_to_pay.csv")
)

print("Dataset shape:", ptp_raw.shape)

print("\nColumn names:")
print(ptp_raw.columns.tolist())

print("\nData types:")
print(ptp_raw.dtypes)

print("\nMissing values:")
display(ptp_raw.isnull().sum())

print("\nFirst five records:")
display(ptp_raw.head())

Dataset shape: (18000, 9)

Column names:
['ptp_id', 'account_id', 'borrower_id', 'event_at', 'agent_id', 'promised_amount', 'promised_date', 'status', 'source']

Data types:
ptp_id                 str
account_id             str
borrower_id            str
event_at               str
agent_id               str
promised_amount    float64
promised_date          str
status                 str
source                 str
dtype: object

Missing values:


ptp_id             0
account_id         0
borrower_id        0
event_at           0
agent_id           0
promised_amount    0
promised_date      0
status             0
source             0
dtype: int64


First five records:


,ptp_id,account_id,borrower_id,event_at,agent_id,promised_amount,promised_date,status,source
0,PTP0000001,ACC0021078,BRW0003789,2026-04-18 21:11:02,AGT0000787,93575.19,2026-05-17 21:11:02,CANCELLED,FIELD
1,PTP0000002,ACC0026620,BRW0010192,2026-07-08 21:49:28,AGT0000763,83558.92,2026-07-25 21:49:28,KEPT,WHATSAPP
2,PTP0000003,ACC0018900,BRW0011197,2026-08-08 08:57:34,AGT0000878,18164.89,2026-09-02 08:57:34,OPEN,SMS
3,PTP0000004,ACC0019107,BRW0009681,2026-01-11 06:04:20,AGT0000535,52829.08,2026-01-24 06:04:20,KEPT,WHATSAPP
4,PTP0000005,ACC0014157,BRW0004907,2026-05-08 01:42:07,AGT0000115,74970.57,2026-05-19 01:42:07,BROKEN,SMS


In [70]:
# Check duplicate records and IDs

print("Exact duplicate rows:", ptp_raw.duplicated().sum())

print("Duplicate PTP IDs:", ptp_raw["ptp_id"].duplicated().sum())

# Check unique values

print("\nPTP status distribution:")
display(ptp_raw["status"].value_counts())

print("\nPTP source distribution:")
display(ptp_raw["source"].value_counts())

# Check amount validity

print("\nPromised amount summary:")
display(ptp_raw["promised_amount"].describe().round(2))

print("\nZero or negative promised amounts:",
      (ptp_raw["promised_amount"] <= 0).sum())

# Check date validity

ptp_raw["event_at_parsed"] = pd.to_datetime(
    ptp_raw["event_at"], errors="coerce"
)

ptp_raw["promised_date_parsed"] = pd.to_datetime(
    ptp_raw["promised_date"], errors="coerce"
)

print("\nInvalid event dates:",
      ptp_raw["event_at_parsed"].isna().sum())

print("Invalid promised dates:",
      ptp_raw["promised_date_parsed"].isna().sum())

print("Promise dates before event dates:",
      (ptp_raw["promised_date_parsed"] <
       ptp_raw["event_at_parsed"]).sum())

Exact duplicate rows: 0
Duplicate PTP IDs: 0

PTP status distribution:


status
BROKEN       4553
CANCELLED    4543
KEPT         4489
OPEN         4415
Name: count, dtype: int64


PTP source distribution:


source
CALL        4571
FIELD       4546
SMS         4449
WHATSAPP    4434
Name: count, dtype: int64


Promised amount summary:


count    18000.00
mean     50233.84
std      28631.75
min        511.99
25%      25441.33
50%      50312.00
75%      74975.03
max      99997.28
Name: promised_amount, dtype: float64


Zero or negative promised amounts: 0

Invalid event dates: 0
Invalid promised dates: 0
Promise dates before event dates: 0


In [72]:
# Load cleaned accounts dataset

accounts = pd.read_csv(
    os.path.join(cleaned_path, "accounts_cleaned.csv")
)

print("Accounts loaded:", accounts.shape)

Accounts loaded: (30000, 13)


In [73]:
# Validate account references

missing_accounts = ~ptp_raw["account_id"].isin(
    accounts["account_id"]
)

# Validate agent references

missing_agents = ~ptp_raw["agent_id"].isin(
    agents_check["agent_id"]
)

print("PTP records with unmatched accounts:", missing_accounts.sum())
print("PTP records with unmatched agents:", missing_agents.sum())

# Validate borrower-account consistency

ptp_account_check = ptp_raw.merge(
    accounts[["account_id", "borrower_id"]],
    on="account_id",
    how="left",
    suffixes=("_ptp", "_account")
)

borrower_mismatch = (
    ptp_account_check["borrower_id_ptp"] !=
    ptp_account_check["borrower_id_account"]
)

print("Borrower-account mismatches:", borrower_mismatch.sum())

PTP records with unmatched accounts: 0
PTP records with unmatched agents: 0
Borrower-account mismatches: 17999


In [74]:
# Check missing borrower IDs in both datasets

print("Missing borrower IDs in PTP:", ptp_raw["borrower_id"].isna().sum())
print("Missing borrower IDs in accounts:", accounts["borrower_id"].isna().sum())

# Inspect sample borrower IDs side by side

print(
    ptp_account_check[
        ["account_id", "borrower_id_ptp", "borrower_id_account"]
    ].head(15).to_string(index=False)
)

# Check data types

print("\nPTP borrower ID type:", ptp_raw["borrower_id"].dtype)
print("Account borrower ID type:", accounts["borrower_id"].dtype)

# Count exact matches, excluding missing values

valid_comparison = ptp_account_check.dropna(
    subset=["borrower_id_ptp", "borrower_id_account"]
)

print(
    "\nExact borrower ID matches:",
    (
        valid_comparison["borrower_id_ptp"] ==
        valid_comparison["borrower_id_account"]
    ).sum()
)

Missing borrower IDs in PTP: 0
Missing borrower IDs in accounts: 455
account_id borrower_id_ptp borrower_id_account
ACC0021078      BRW0003789          BRW0004120
ACC0026620      BRW0010192          BRW0001002
ACC0018900      BRW0011197          BRW0000865
ACC0019107      BRW0009681          BRW0001221
ACC0014157      BRW0004907          BRW0011936
ACC0020223      BRW0004429          BRW0001281
ACC0018006      BRW0008673          BRW0008912
ACC0011737      BRW0007288          BRW0002510
ACC0021229      BRW0007682          BRW0009639
ACC0028901      BRW0009589          BRW0007140
ACC0015304      BRW0011294          BRW0009462
ACC0014684      BRW0003496          BRW0002726
ACC0022800      BRW0003170          BRW0002542
ACC0022482      BRW0004731          BRW0008464
ACC0000494      BRW0002652          BRW0000403

PTP borrower ID type: str
Account borrower ID type: str

Exact borrower ID matches: 1


In [75]:
# Check whether PTP borrower IDs are reused across accounts

print(
    "Unique PTP borrower IDs:",
    ptp_raw["borrower_id"].nunique()
)

print(
    "Unique account borrower IDs:",
    accounts["borrower_id"].nunique()
)

# Check whether PTP borrower IDs exist in the accounts dataset

account_borrower_ids = set(accounts["borrower_id"].dropna())

ptp_borrower_ids = set(ptp_raw["borrower_id"].dropna())

print(
    "PTP borrower IDs absent from account master:",
    len(ptp_borrower_ids - account_borrower_ids)
)

# Check missing borrower IDs in the account master for PTP-linked accounts

print(
    "PTP records linked to accounts with missing master borrower ID:",
    ptp_account_check["borrower_id_account"].isna().sum()
)

Unique PTP borrower IDs: 9299
Unique account borrower IDs: 10943
PTP borrower IDs absent from account master: 835
PTP records linked to accounts with missing master borrower ID: 293


In [76]:
# Create a working copy of PTP data

ptp_cleaned = ptp_raw.copy()

# Convert date columns

ptp_cleaned["event_at"] = pd.to_datetime(
    ptp_cleaned["event_at"], errors="coerce"
)

ptp_cleaned["promised_date"] = pd.to_datetime(
    ptp_cleaned["promised_date"], errors="coerce"
)

# Add account and agent validation flags

ptp_cleaned["account_reference_valid"] = (
    ptp_cleaned["account_id"].isin(accounts["account_id"])
)

ptp_cleaned["agent_reference_valid"] = (
    ptp_cleaned["agent_id"].isin(agents_check["agent_id"])
)

# Add borrower mapping validation flag

ptp_cleaned = ptp_cleaned.merge(
    accounts[["account_id", "borrower_id"]],
    on="account_id",
    how="left",
    suffixes=("", "_account")
)

ptp_cleaned["borrower_mapping_mismatch"] = (
    ptp_cleaned["borrower_id"] !=
    ptp_cleaned["borrower_id_account"]
)

# Save cleaned dataset

ptp_cleaned.to_csv(
    os.path.join(cleaned_path, "ptp_cleaned.csv"),
    index=False
)

print("Cleaned PTP shape:", ptp_cleaned.shape)
print("Saved successfully: ptp_cleaned.csv")
print(
    "Borrower mapping mismatches:",
    ptp_cleaned["borrower_mapping_mismatch"].sum()
)

Cleaned PTP shape: (18000, 15)
Saved successfully: ptp_cleaned.csv
Borrower mapping mismatches: 17999
